# 🎯 23 · Parameter estimation and maximum likelihood
### Statistical inference with parametric models, choosing a model, the likelihood and the log-likelihood, maximum-likelihood estimation by calculus and by computer (Bernoulli, normal, Poisson, zero-inflated counts), and the properties of the MLE

**BiologyToMath** · Part F — Statistics: from data to estimates · Module 23 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/23_maximum_likelihood.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`, `scipy.optimize`), SymPy, pandas, Matplotlib |
| **Prerequisites** | [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) (i.i.d. data, sample mean, standard error, CLT), [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb) (derivatives, logarithms, maxima), [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) (Bernoulli, binomial, Poisson, normal, exponential) and [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) (Bayes' rule) |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Statistics in a nutshell (recap) and what a statistic is; statistical inference (learning): data generator, observations, experience, parameter estimation, model; the coin-flip simulator; parametric models $\mathcal F = \{f(x\mid\theta): \theta\in\Theta\}$ and why we study them; requirements (generalise, use prior knowledge, scale); selecting a parametric model (discrete/continuous, uni/multivariate, prior knowledge, mean–variance relationship); the data likelihood $p(\mathcal D\mid\theta) = \prod_i p(X_i\mid\theta)$ via Bayes' theorem; the log-likelihood; the maximum-likelihood estimate and the calculus recipe (critical points, second derivative, boundary maxima); Example 1 (1,1,1,0,0,0,0,0,1,0 → 0.4); MLE for the Bernoulli ($s/n$), normal ($\bar x$ and the $1/n$ variance) and Poisson ($\bar x$) distributions; summary |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All data are **SIMULATED** with known truth: coin flips and base calls, single-cell counts (toolkit `simulate_cell_counts`), Hardy–Weinberg genotypes (`simulate_genotypes`), expression values, mutation counts, binding residence times, foci positions and a regulator–target data set. The ten-flip sequence of Example 1 is typed in. Rates (mutation rate, off-rate) are illustrative orders of magnitude, not measurements.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Describe** statistical inference as choosing a member $\hat f$ of a parametric model $\mathcal F = \{f(x\mid\theta) :
   \theta\in\Theta\}$ from data, and **draw** the data-generator → observations → estimation → model framework.
2. **Choose** a sensible parametric family for a biological measurement (counts, proportions, continuous values) and
   **use** the mean–variance relationship to check it.
3. **Write** the likelihood $p(\mathcal D\mid\theta) = \prod_i p(x_i\mid\theta)$ and log-likelihood of i.i.d. data, and
   **explain** why we maximise the log (underflow; products become sums).
4. **Compute** maximum-likelihood estimates with the calculus recipe — derivative, critical point, second-derivative check,
   boundary maxima — by hand and with SymPy.
5. **Derive** the MLEs of the Bernoulli ($\hat\alpha = s/n$), normal ($\hat\mu = \bar x$, $\hat\sigma^2 = \frac1n\sum(x_i - \bar x)^2$)
   and Poisson ($\hat\lambda = \bar x$) models line by line, and **prove** that $\hat\sigma^2$ is biased.
6. **Fit** models without closed-form MLEs (a zero-inflated Poisson for single-cell dropout) by numerical optimisation of the
   negative log-likelihood.
7. **Demonstrate** consistency, invariance and asymptotic normality of the MLE, and **compute** a standard error from the
   curvature of the log-likelihood.
8. **Apply** MLE to biology: allele frequencies, cell-type expression rates, mutation rates, binding kinetics, and
   least-squares regression.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, simulate single-cell counts and genotypes | 5 min |
| 1 · Statistical inference | data generators vs models; the coin-flip (base-call) simulator; parametric models; the framework | 20 min |
| 2 · Choosing a parametric model | why parametric; what to consider; the mean–variance check on single-cell counts | 15 min |
| 3 · The likelihood | Bayes' theorem view; the i.i.d. product; likelihood ≠ probability of θ; the log-likelihood and underflow | 20 min |
| 4 · Maximum likelihood | definition; the calculus recipe; Example 1 two ways; boundary maxima; "is the coin unfair?" | 25 min |
| 5 · Bernoulli MLE | $\hat\alpha = s/n$ in general; allele frequencies from genotypes | 15 min |
| 6 · Normal MLE | $\hat\mu$, $\hat\sigma^2$ line by line; the likelihood surface; the bias of $\hat\sigma^2$ | 25 min |
| 7 · Poisson MLE (and exposures, waiting times) | expression rates per cell type; mutation rates; binding residence times | 20 min |
| 8 · MLE by computer | `scipy.optimize.minimize`; a zero-inflated Poisson for dropout | 15 min |
| 9 · Properties of the MLE | consistency, invariance, asymptotic normality, standard errors; least squares = MLE | 20 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

A single-cell RNA-seq experiment has profiled 440 immune and stromal cells from a tissue. For the T-cell marker *CD3E*
the 160 T cells contain 7, 15, 5, 6, … captured molecules each. Before any downstream analysis — clustering, differential
expression, a cell-type classifier — someone has to answer a deceptively simple question: **what is the expression level
of *CD3E* in T cells?** Not "what are these 160 numbers", but "what single number (a *parameter*) describes the process that
generated them, and how sure are we of it?"

The same question has a hundred biological faces. What is the frequency of a risk allele, given 500 genotyped people? What
is the mutation rate per base per generation, given a few mutation-accumulation lines? How long does a transcription factor
stay bound to DNA, given 200 imaged binding events? In every case we (1) **choose a model** — a family of probability
distributions indexed by an unknown parameter $\theta$ — and (2) **estimate $\theta$** from the data. The most frequently used
estimation principle in all of science is **maximum likelihood**: choose the $\theta$ under which the observed data were most
probable. This module derives it from scratch, applies it to Bernoulli, normal and Poisson models, and shows how to use a
computer when calculus runs out.

| Biological question | Model $f(x\mid\theta)$ | MLE | Where |
|---|---|---|---|
| Is this coin / test cross / sequencer fair? | Bernoulli($\alpha$) | $\hat\alpha = s/n$ | 1, 4, 5 |
| Risk-allele frequency from 500 genotypes | Binomial(2, $p$) per person | $\hat p = (2n_2 + n_1)/2n$ | 5 |
| Expression of a gene in a log scale | Normal($\mu, \sigma^2$) | $\bar x$, $\frac1n\sum(x_i - \bar x)^2$ | 6 |
| Molecules of *CD3E* per T cell | Poisson($\lambda$) | $\hat\lambda = \bar x$ | 7 |
| Mutations per base per generation | Poisson($\mu L_i$) | $\sum k_i / \sum L_i$ | 7 |
| Residence time of a DNA-bound factor | Exponential($k$) | $\hat k = 1/\bar x$ | 7 |
| Counts with technical dropout | zero-inflated Poisson($\pi, \lambda$) | numerical | 8 |
| Target gene as a function of its regulator | Normal($\beta_0 + \beta_1 x, \sigma^2$) | least squares | 9 |

> 🧪 **Simulated data, with a known truth.** Every data set is generated by code: the single-cell counts by the toolkit's
> `simulate_cell_counts` (four cell types, Poisson counts with known rates λ for 40 genes), genotypes by
> `simulate_genotypes` (Hardy–Weinberg, known allele frequency). Because we know the true parameters, we can check every
> estimate against the truth — and watch what happens over thousands of imaginary repeat experiments.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import stats
from scipy.optimize import minimize

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). They simulate single-cell RNA counts for T cells, B cells, macrophages and fibroblasts (with the true Poisson rate of every gene in every cell type), list the marker genes of each cell type, and simulate Hardy–Weinberg genotypes at one SNP. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd


CELL_TYPE_MARKERS = {
    "T cell": ["CD3E", "CD3D", "IL7R"],
    "B cell": ["CD79A", "MS4A1", "CD19"],
    "macrophage": ["CD68", "LYZ", "CSF1R"],
    "fibroblast": ["COL1A1", "COL1A2", "DCN"],
}


def simulate_cell_counts(n_cells=(160, 120, 90, 70), cell_types=("T cell", "B cell", "macrophage", "fibroblast"),
                         n_background: int = 28, overdispersion: float = 0.0, seed: int = 680) -> dict:
    """SIMULATED single-cell RNA counts (molecules of each gene detected in each cell).

    Each cell type has a true mean count λ for every gene: its three ``CELL_TYPE_MARKERS`` genes are
    high (λ ≈ 6–15) in that type and low (≈ 0.1–0.5) elsewhere; ``n_background`` other genes share
    similar λ (≈ 0.2–4) in all types.  Counts are Poisson(λ); with ``overdispersion`` φ > 0 each
    cell's λ is first multiplied by a Gamma(1/φ, φ) factor (mean 1), which makes counts negative-
    binomial with variance λ + φλ² — closer to real data.

    Returns a dict with ``counts`` (cells × genes, int), ``labels`` (cell type of each cell),
    ``genes``, ``rates`` (DataFrame cell types × genes of true λ) and ``source``.
    """
    rng = np.random.default_rng(seed)
    markers = [g for t in cell_types for g in CELL_TYPE_MARKERS[t]]
    genes = markers + [f"bg{i:02d}" for i in range(1, n_background + 1)]
    rates = np.zeros((len(cell_types), len(genes)))
    bg = rng.uniform(0.2, 4.0, n_background)
    for t, ctype in enumerate(cell_types):
        for j, g in enumerate(markers):
            rates[t, j] = rng.uniform(6.0, 15.0) if g in CELL_TYPE_MARKERS[ctype] else rng.uniform(0.1, 0.5)
        rates[t, len(markers):] = bg * rng.uniform(0.8, 1.25, n_background)
    labels, counts = [], []
    for t, (ctype, n) in enumerate(zip(cell_types, n_cells)):
        lam = np.tile(rates[t], (n, 1))
        if overdispersion > 0:
            lam = lam * rng.gamma(1.0 / overdispersion, overdispersion, size=lam.shape)
        counts.append(rng.poisson(lam))
        labels += [ctype] * n
    return {"counts": np.vstack(counts), "labels": np.array(labels), "genes": genes,
            "rates": pd.DataFrame(rates, index=list(cell_types), columns=genes),
            "source": "SIMULATED single-cell counts (BiologyToMath simulate_cell_counts)"}


def simulate_genotypes(n: int, p: float, inbreeding: float = 0.0, seed: int = 1866) -> np.ndarray:
    """SIMULATED genotypes of ``n`` diploid individuals at one SNP, coded 0/1/2 = number of copies of
    the allele with frequency ``p``.  With ``inbreeding`` F = 0 the genotype frequencies follow
    Hardy–Weinberg proportions ((1−p)², 2p(1−p), p²); F > 0 moves individuals from heterozygotes to
    homozygotes: P(1) = 2p(1−p)(1−F)."""
    rng = np.random.default_rng(seed)
    q = 1.0 - p
    probs = np.array([q * q + inbreeding * p * q, 2 * p * q * (1 - inbreeding), p * p + inbreeding * p * q])
    return rng.choice(3, size=n, p=probs / probs.sum())

print("🧰 toolkit loaded:", ", ".join(sorted(['CELL_TYPE_MARKERS', 'simulate_cell_counts', 'simulate_genotypes'])))

In [ ]:
# ── 0.1 The data of this module (SIMULATED, fixed seeds) ──
sc = simulate_cell_counts()                                   # 440 cells × 40 genes, Poisson counts
counts, labels, genes, true_rates = sc["counts"], sc["labels"], sc["genes"], sc["rates"]
print(sc["source"], "|", counts.shape[0], "cells ×", counts.shape[1], "genes")
print("cells per type:", {t: int(np.sum(labels == t)) for t in CELL_TYPE_MARKERS})
cd3e_t = counts[labels == "T cell", genes.index("CD3E")]
print("CD3E molecules in the first 12 T cells:", cd3e_t[:12].tolist(), "…")
genotypes = simulate_genotypes(500, 0.3, seed=7)              # 500 people, one SNP, true allele frequency 0.3
print("genotype counts (0, 1, 2 copies of the allele):", np.bincount(genotypes, minlength=3).tolist())

🔍 **What just happened?** We generated the two main data sets: 440 single cells (160 T cells, 120 B cells, 90
macrophages, 70 fibroblasts) with counts for 40 genes, and the genotypes of 500 people at one SNP (231, 226 and 43 people carry 0,
1 or 2 copies of the allele). The first T cells carry 7, 15, 5, 6, 15, 7, … molecules of *CD3E*. The simulator also returned the true
rates (`true_rates`), which we keep hidden until we want to check our estimates.

---
## 1 · Statistical inference: data generators and models

> 📍 **Where we are.** Module 22 introduced statistics in a nutshell — collect data (after forming hypotheses), describe
> them, analyse them — and two guiding questions: "Is it fair?" and "Is it normally distributed?". It also defined a
> **statistic**: anything computable from the data, itself a random variable. Modules 16–21 were about the models
> themselves. Now we connect the two: how do we make the models we have been studying **match the observations we have made
> about the world**? That task is **statistical inference** (also called **learning**).

**The data generator.** Think back to a simulator that usually outputs "heads" or "tails" but, once in every 1,000 flips,
outputs a nonsense number. The simulator is a **data generator**: it produces **observations** we can see and measure, but
we cannot see inside it — its true parameters (the probability of heads, the probability of a glitch) are hidden. A DNA
sequencer is exactly such a generator: it calls A, C, G or T, and once in a while it outputs "N" (no call) or garbage. Let us
build one and look only at what it emits.

In [ ]:
# ── 1.1 A data generator we can only observe: a coin-flip simulator with rare nonsense outputs (SIMULATED) ──
def glitchy_coin_simulator(n, rng, p_heads=0.5, glitch_rate=0.001):
    """SIMULATED data generator: returns n outputs, each 'H' (prob. p_heads·(1 − glitch_rate)), 'T', or — about once
    in 1/glitch_rate outputs — a nonsense number. Its parameters are what inference tries to recover."""
    out = np.where(rng.random(n) < p_heads, "H", "T").astype(object)
    glitches = rng.random(n) < glitch_rate
    out[glitches] = [f"{v:.3f}" for v in rng.normal(0, 1000, glitches.sum())]
    return out


rng_gen = np.random.default_rng(SEED + 11)
observations = glitchy_coin_simulator(5000, rng_gen)
print("first 40 outputs:", " ".join(observations[:40]))
n_H, n_T = np.sum(observations == "H"), np.sum(observations == "T")
odd = [o for o in observations if o not in ("H", "T")]
print(f"5,000 outputs: {n_H} H, {n_T} T, {len(odd)} nonsense values {odd}")
print(f"estimated P(heads | a coin outcome) = {n_H}/{n_H + n_T} = {n_H / (n_H + n_T):.4f};  "
      f"estimated glitch rate = {len(odd)}/5000 = {len(odd) / 5000:.4f}")

🔍 **What just happened?** We only see the outputs: 2,531 heads, 2,466 tails and 3 nonsense values among 5,000. From them we can
estimate the hidden parameters — the fraction of heads among genuine flips (0.5065, truth 0.5) and the glitch rate
(0.0006, truth 0.001). Notice that the second estimate rests on just a handful of events: rare-event rates are hard to
estimate, a theme of Module 24.

**Models.** On the other side, we have a set of **models**: mathematical representations of *how we think the data might have
been generated*. Each model has its own set of **parameters** — like the probability of heads of a biased coin. "Bernoulli
with unknown $\alpha$" is one model of the H/T outputs; "categorical over {H, T, glitch}" is a richer one.

**Statistical inference** is the task of
* using the data (observations)
* to estimate or "learn" the parameters of the model,
* so that the model better matches the observed data.

It is about bridging the gap between theory (models) and reality (data).

**The formal set-up.** We have a data set $\mathcal D = X_1, X_2, \dots, X_n$ of i.i.d. random variables from some unknown
distribution. Let $\mathcal F$ be a **statistical model**: a *set* of probability distributions. We focus on **parametric
models**, sets indexed by a finite number of parameters:

$$\mathcal F = \{\, f(x\mid\theta) : \theta\in\Theta \,\}, \qquad \theta = (\theta_1, \theta_2, \dots, \theta_k) .$$

*In words:* the model is the whole family of densities (or PMFs) $f(x\mid\theta)$ you get by letting the unknown parameter
vector $\theta$ range over its allowed set $\Theta$, the **parameter space**. Example: the Bernoulli model is
$\{f(x\mid\alpha) = \alpha^x(1-\alpha)^{1-x} : \alpha\in[0,1]\}$ — one parameter, $\Theta = [0, 1]$; the normal model has
$\theta = (\mu, \sigma^2)$ and $\Theta = \mathbb R\times(0,\infty)$.

**Objective.** Find a distribution $\hat f\in\mathcal F$ that models the phenomenon well — which means estimating $\theta$.
We want to do this in a way that has (1) the ability to **generalise** well (the fitted model should describe *new* data from
the same generator, not just these); (2) the ability to **incorporate prior knowledge and assumptions**; and (3) the
ability to **scale** (to many parameters and large data). The whole pipeline:

In [ ]:
# ── 1.2 The framework of parametric estimation, drawn ──
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")


def box(x, y, text, dashed=False, color="0.95"):
    """Draw a labelled box centred at (x, y)."""
    ax.text(x, y, text, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="square,pad=0.6", fc=color, ec="0.3", ls="--" if dashed else "-"))


def arrow(x1, y1, x2, y2, label="", dashed=False):
    """Draw an arrow from (x1, y1) to (x2, y2) with an optional label."""
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color="0.3", ls="--" if dashed else "-"))
    if label:
        ax.text((x1 + x2) / 2 + 0.1, (y1 + y2) / 2, label, fontsize=8, color="tab:blue")


ax.text(4.6, 5.6, "Observations", ha="center", fontsize=9, style="italic")
ax.text(8.0, 5.6, "Knowledge + assumptions", ha="center", fontsize=9, style="italic")
box(1.4, 4.8, "Data generator\n(unknown)", dashed=True, color="#fde0dd")
box(4.6, 4.8, "Data set")
box(8.0, 4.8, "Experience")
box(6.3, 2.8, "Parameter estimation\n(optimisation)", color="#e0ecf4")
box(6.3, 0.8, r"Model  $f(x \mid \hat\theta)$", color="#e5f5e0")
arrow(2.5, 4.8, 3.9, 4.8, dashed=True)
arrow(4.6, 4.3, 5.7, 3.3, "D = X₁, …, Xₙ")
arrow(8.0, 4.3, 6.9, 3.3, r"pick $\hat f \in \mathcal{F}$")
arrow(6.3, 2.3, 6.3, 1.3, "estimate θ ∈ Θ")
ax.text(1.4, 2.2, "Model inference =\nobservations\n+ knowledge and assumptions\n+ optimisation", ha="center",
        fontsize=9, color="0.25")
ax.set_title("Statistical inference: from an unseen data generator to a fitted model")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The diagram is the map of Modules 23–27. The **data generator** (left, dashed) is unknown — the
biology itself, or the simulator. It produces the **data set** $\mathcal D = X_1,\dots,X_n$ (observations). Separately, our
**experience** — knowledge and assumptions — tells us which family $\mathcal F$ to pick ("counts → Poisson"). **Parameter
estimation** combines the two by **optimisation** and outputs a **model** with estimated parameters $\hat\theta$. Model
inference = observations + knowledge and assumptions + optimisation. This module is about the optimisation step;
Section 2 is about the "experience" step.

✏️ **Your turn.** Rerun 1.1 with `glitch_rate=0.01` and 500 outputs, ten times with different seeds. How much does the
estimated glitch rate vary? (Module 22: a proportion from $n$ trials has SE $\sqrt{p(1-p)/n}$ — compute it.)

### 📘 Lecture notes — statistical inference and parametric models

**Statistical inference (learning)** uses the data (observations) to estimate or "learn" the parameters of a model, so
that the model better matches the observed data — bridging theory (models) and reality (data). A **data generator** (e.g. a
coin-flip simulator that occasionally outputs nonsense) produces observations we can see; its inner parameters are unknown.
**Models** are mathematical representations of how we think the data were generated, each with its own parameters.

**Formal set-up.** $\mathcal D = X_1,\dots,X_n$ i.i.d. from an unknown distribution; a **statistical model** $\mathcal F$
is a set of distributions; a **parametric model** is $\mathcal F = \{f(x\mid\theta) : \theta\in\Theta\}$ with unknown
parameters $\theta = \{\theta_1, \dots, \theta_k\}$ in the **parameter space** $\Theta$.

**Objective.** Find $\hat f\in\mathcal F$ that models the phenomenon well, i.e. estimate $\theta$. **Requirements:** the
ability to generalise well; to incorporate prior knowledge and assumptions; to scale.

**Framework.** Data generator (unknown) ⇢ data set $\mathcal D$ (observations); experience (knowledge + assumptions) ⇒
pick $\hat f\in\mathcal F$; both → **parameter estimation** (optimisation) → estimate $\theta\in\Theta$ → **model**.
*Model inference = observations + knowledge and assumptions + optimisation.*

> 🎓 **Exam tip.** For any estimation problem, name the three ingredients explicitly: the data $\mathcal D$, the family
> $\mathcal F$ (with $\theta$ and $\Theta$), and the optimisation criterion (here: maximum likelihood).

---
## 2 · Choosing a parametric model

> 📍 **Where we are.** Estimation needs a family $\mathcal F$ first. Where does it come from, and why restrict ourselves to
> parametric families at all?

**Why study parametric models?** Real data generators are rarely *exactly* Bernoulli or normal, and methods that assume
no particular family (nonparametric methods: the bootstrap of Module 22, rank tests, kernel density estimates) exist.
Despite their limitations, parametric models remain valuable for two important reasons:

1. **Useful approximations in specific contexts.** In many domains prior knowledge suggests that certain parametric families
   are reasonable: molecules captured from a cell behave like Poisson counts; a trait built from many small effects is close
   to normal (Module 22); the number of carriers among $n$ people is binomial.
2. **A conceptual foundation for inference.** Core statistical ideas — likelihood, maximum-likelihood estimation (MLE),
   confidence intervals — are developed within parametric frameworks. These foundational concepts motivate and inform the
   development of nonparametric and semiparametric methods.

**Selecting a parametric model: things to consider.**

| Question | If … | consider |
|---|---|---|
| Discrete or continuous? | counts, categories, yes/no | Bernoulli, binomial, categorical, Poisson, negative binomial |
| | real-valued measurements | normal, exponential, gamma, log-normal |
| Univariate or multivariate features? | one gene / many genes at once | a univariate family / a multivariate normal (Module 19) |
| Prior knowledge | count data in intervals (molecules per cell, mutations per genome, cases per week) | **Poisson** |
| | real-valued, bell-shaped (height, log-expression) | **normal** |
| | waiting time of a memoryless process | exponential |
| Evidence from exploratory analysis | e.g. the **relationship between mean and variance** | Poisson: variance = mean; over-dispersed counts: variance > mean |

The last row is a powerful check. A Poisson variable has variance equal to its mean (Module 21), so if counts of many
genes are Poisson, a plot of each gene's variance against its mean should hug the line *variance = mean*. Let us check
our single-cell data, and a second data set from the same simulator with biological over-dispersion switched on.

In [ ]:
# ── 2.1 Exploratory check of the model: variance vs mean of every gene within T cells (SIMULATED) ──
sc_over = simulate_cell_counts(overdispersion=0.5)            # same design, cell-to-cell variation in each rate
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (title, data) in zip(axes, [("Poisson counts (overdispersion = 0)", sc), ("over-dispersed counts (φ = 0.5)", sc_over)]):
    t_cells = data["counts"][data["labels"] == "T cell"]
    m, v = t_cells.mean(axis=0), t_cells.var(axis=0, ddof=1)
    grid = np.logspace(-1.3, 1.3, 50)
    ax.loglog(m, v, "o", color="tab:blue", ms=4, label="one gene (40 genes)")
    ax.loglog(grid, grid, "k--", lw=1, label="variance = mean (Poisson)")
    ax.loglog(grid, grid + 0.5 * grid ** 2, ":", color="tab:red", lw=1.5, label="variance = mean + 0.5·mean²")
    ax.set_xlabel("mean count per T cell (log scale)"); ax.set_title(title); ax.legend(fontsize=7)
    ratio = (v / m)[m > 1]
    print(f"{title:38s}: variance/mean for genes with mean > 1: median {np.median(ratio):.2f}, max {ratio.max():.2f}")
axes[0].set_ylabel("variance across T cells (log scale)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the left panel every gene sits on the line *variance = mean*: the median variance-to-mean ratio
of the well-expressed genes is 1.01, as a Poisson model predicts. In the right panel the same genes have variances far
above their means (median ratio 2.27), following *variance = mean + 0.5·mean²* — the signature of the
**negative binomial** (gamma–Poisson) model that real single-cell data usually require. The exploratory plot chose the model
family for us: Poisson for the left data set, something over-dispersed for the right one. In the rest of this module we use
the Poisson data set, for which the Poisson family is correct.

✏️ **Your turn.** Make the same plot for the macrophages, and for all 440 cells pooled together (ignoring cell type). Why do
the pooled marker genes look over-dispersed even in the Poisson data set? (Hint: a mixture of Poissons with different
rates.)

---
## 3 · The likelihood

> 📍 **Where we are.** We have data and a family $f(x\mid\theta)$. We need a *score* that says how well each $\theta$ explains
> the data. Bayes' theorem (Module 17) points to it.

**Bayes' theorem, written for hypotheses and data.**

$$p(\text{hypothesis}\mid\text{data}) = \frac{p(\text{data}\mid\text{hypothesis})\,p(\text{hypothesis})}{p(\text{data})},
\qquad\text{i.e.}\qquad P(h\mid\mathcal D) = \frac{P(\mathcal D\mid h)\,P(h)}{P(\mathcal D)} .$$

*In words:* how plausible a hypothesis (a parameter value) is after seeing the data — the **posterior** $P(h\mid\mathcal D)$ —
is the **likelihood** $P(\mathcal D\mid h)$ (how probable the data are under that hypothesis) times the **prior** $P(h)$ (how
plausible it was before), divided by the **evidence** $P(\mathcal D)$, the overall probability of the data. When all the
probabilities on the right are known exactly, we can compute the one on the left exactly. If we can figure out all the values
on the right, we learn how good our model parameters are given the data.

The most complicated part of the right-hand side is $p(\text{data}\mid\text{hypothesis})$, in our notation $p(\mathcal D\mid\theta)$.
The prior needs outside knowledge (Module 24 is all about it); the evidence $p(\mathcal D)$ does not depend on $\theta$ at
all. Maximum likelihood keeps only the likelihood: it ignores the prior — equivalently, treats every $\theta$ as equally
plausible beforehand — and drops $p(\mathcal D)$, which cannot change which $\theta$ wins.

**The likelihood of i.i.d. data.** Let $\mathcal D = X_1,\dots,X_n$ be i.i.d. samples from a distribution $h\in\mathcal F$ with
parameter $\theta$. The **likelihood of the data** is the probability (or density) of $\mathcal D$ given $\theta$; because
the $X_i$ are independent it factorises into a product (Module 19):

$$p(\mathcal D\mid\theta) = p(X_1, X_2, \dots, X_n\mid\theta) = \prod_{i=1}^{n} p(X_i\mid\theta) .$$

*In words:* the probability of the whole data set is the product of the probabilities of the individual observations —
"independent" is what lets us multiply, "identically distributed" is what lets every factor use the same $\theta$.

🧮 **Math refresher — Π, and functions of the parameter.** $\prod_{i=1}^n a_i = a_1\cdot a_2\cdots a_n$ is repeated
multiplication, as $\sum$ is repeated addition. Once the data are observed they are fixed numbers, so $p(\mathcal D\mid\theta)$
is a function of $\theta$ alone: we write $L(\theta) = p(\mathcal D\mid\theta)$ and call it the **likelihood function**.

**Example 1 (the data of the next section).** To learn the probability of heads $\alpha$ of a new coin — or the frequency of
an antibiotic-resistance gene among bacterial isolates — we observe $\mathcal D = 1,1,1,0,0,0,0,0,1,0$ (1 = heads/resistant).
With $f(x\mid\alpha) = \alpha^x(1-\alpha)^{1-x}$,

$$L(\alpha) = \prod_{i=1}^{10}\alpha^{x_i}(1-\alpha)^{1-x_i} = \alpha^{\sum_i x_i}(1-\alpha)^{\sum_i(1 - x_i)} = \alpha^4(1-\alpha)^6 .$$

*In words:* multiply $\alpha$ once for each 1 and $(1-\alpha)$ once for each 0; exponents add up, so only the counts (4 ones,
6 zeros) matter, not their order.

In [ ]:
# ── 3.1 The likelihood of Example 1 as a function of α ──
D1 = np.array([1, 1, 1, 0, 0, 0, 0, 0, 1, 0])
alpha_grid = np.linspace(0.0005, 0.9995, 1999)


def bernoulli_likelihood(alpha, x):
    """L(α) = Π α^x_i (1 − α)^(1 − x_i) for 0/1 data x (vectorised over an array of α values)."""
    alpha = np.asarray(alpha, dtype=float)[..., None]
    return np.prod(alpha ** x * (1 - alpha) ** (1 - x), axis=-1)


L1 = bernoulli_likelihood(alpha_grid, D1)
for a in [0.2, 0.4, 0.5, 0.8]:
    print(f"L({a}) = {a}^4 × {1 - a:.1f}^6 = {bernoulli_likelihood(a, D1):.6f}")
print(f"area under L(α) from 0 to 1 ≈ {np.sum(L1) * (alpha_grid[1] - alpha_grid[0]):.6f} "
      f"(exactly 4!·6!/11! = {1 / 2310:.6f}) — not 1")
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(alpha_grid, L1, color="tab:blue")
ax.axvline(0.4, color="k", ls="--", lw=1, label="highest point: α = 0.4")
ax.set_xlabel("probability of heads (or of resistance) α"); ax.set_ylabel("likelihood L(α)")
ax.set_title("Example 1, D = 1,1,1,0,0,0,0,0,1,0:  L(α) = α⁴(1 − α)⁶"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The likelihood is a hill over $\alpha\in[0,1]$. At $\alpha = 0.5$ the data have probability
$0.5^{10} = 0.000977$; at $\alpha = 0.4$ a little more, 0.001194; at 0.2 or 0.8 much less. The hill's peak is at $\alpha = 0.4$.
The area under $L(\alpha)$ is $1/2310$, not 1.

> ⚠️ **Common confusion — the likelihood is not a probability distribution over θ.** $L(\alpha)$ is the probability *of the
> data* for each fixed $\alpha$; read as a function of $\alpha$ it need not integrate to 1 (here $1/2310$). "The likelihood
> of $\alpha = 0.4$ is 0.0012" does **not** mean "the probability that $\alpha = 0.4$ is 0.0012". Turning a likelihood into a
> probability over $\theta$ needs a prior — Module 24.

### 3.2 The log-likelihood

When we multiply many probabilities the product becomes astronomically small — smaller than the smallest number a computer
can store (about $10^{-308}$ for 64-bit floats), at which point it is rounded to exactly 0, called **underflow**. So we
usually take the logarithm of the likelihood, which also turns the product into a sum:

$$\log p(X_1, X_2, \dots, X_n\mid\theta) = \log\prod_{i=1}^n p(X_i\mid\theta) = \sum_{i=1}^{n}\log p(X_i\mid\theta) \qquad (X_1,\dots,X_n \text{ i.i.d.}) .$$

*In words:* the log of a product is the sum of the logs; the **log-likelihood** $\ell(\theta) = \log L(\theta)$ adds one term per
observation. Because $\log$ is strictly increasing, $L$ and $\ell$ are largest at the same $\theta$.

🧮 **Math refresher — logarithm rules** (Module 14; $\log$ = natural log). $\log(ab) = \log a + \log b$;
$\log(a/b) = \log a - \log b$; $\log a^k = k\log a$; $\log e^x = x$; $\frac{d}{dx}\log x = \frac1x$; and by the chain rule
$\frac{d}{dx}\log g(x) = \frac{g'(x)}{g(x)}$, e.g. $\frac{d}{d\alpha}\log(1-\alpha) = \frac{-1}{1-\alpha}$.

In [ ]:
# ── 3.2 Underflow: the likelihood of 2,000 observations is "0" in floating point, the log-likelihood is fine ──
rng_u = np.random.default_rng(SEED + 32)
expr2000 = rng_u.normal(5.0, 1.2, 2000)                     # SIMULATED log2 expression of a gene in 2,000 cells
for mu_try in [4.5, 5.0, 5.5]:
    dens = stats.norm.pdf(expr2000, mu_try, 1.2)
    print(f"μ = {mu_try}: product of 2,000 densities = {np.prod(dens)!r:8}   sum of log densities = {np.sum(np.log(dens)):10.2f}")
print(f"0.5 ** 2000 = {0.5 ** 2000!r}  but  2000 · log 0.5 = {2000 * np.log(0.5):.2f}  (i.e. 10^{2000 * np.log10(0.5):.0f})")
print("smallest positive float64:", np.finfo(float).tiny, "(denormals reach ~5e-324)")

🔍 **What just happened?** For 2,000 cells the product of the densities is exactly `0.0` for every $\mu$ we try — the
computer can no longer tell $\mu = 4.5$ from $\mu = 5$ — while the log-likelihoods (−3385.5, −3221.1 and −3404.0) clearly rank the three values
(the true $\mu = 5$ wins). A likelihood of $0.5^{2000}$ is about $10^{-602}$, far below the smallest representable number. In
practice **always compute log-likelihoods**, summing `logpdf`/`logpmf` values instead of multiplying `pdf` values.

✏️ **Your turn.** How many fair-coin flips does it take before `0.5 ** n` underflows to 0? (Try `n = 1000, 1074, 1075`.)

### 📘 Lecture notes — the data likelihood and the log-likelihood

**Bayes' theorem** $P(h\mid\mathcal D) = \dfrac{P(\mathcal D\mid h)P(h)}{P(\mathcal D)}$: posterior = likelihood × prior / evidence
($P(\mathcal D)$ is the overall probability of the data, the "data distribution"). If all terms on the right are known exactly,
the posterior is known exactly. Maximum likelihood uses only the **likelihood**; the prior is ignored (treated as flat) and
$P(\mathcal D)$ does not depend on $\theta$.

**Likelihood of the data.** For $\mathcal D = X_1,\dots,X_n$ i.i.d. from $h\in\mathcal F$ with parameter $\theta$:

$$P(\mathcal D\mid h) = P(X_1,\dots,X_n\mid\theta) = \prod_{i=1}^n P(X_i\mid\theta), \qquad
\log P(\mathcal D\mid h) = \sum_{i=1}^n \log P(X_i\mid\theta) .$$

*In words:* independence turns the joint probability into a product, and the log turns the product into a sum.

**Why logs.** Products of many probabilities underflow (2,000 normal densities → `0.0`; $0.5^{2000}\approx 10^{-602}$); sums of
logs do not; $\log$ is increasing, so the maximiser is unchanged.

**Common confusion.** $L(\theta)$ is a function of $\theta$ but a probability *of the data*: its integral over $\theta$ need
not be 1 (Example 1: $1/2310$).

> 🎓 **Exam tip.** Write the likelihood in three lines: (1) $\prod_i f(x_i\mid\theta)$ with the PMF/PDF plugged in;
> (2) collect powers / exponents; (3) take logs. Most MLE exam questions are then one derivative away.

---
## 4 · The maximum-likelihood estimate

> 📍 **Where we are.** We can now score every $\theta$ by $L(\theta)$. The natural estimate is the best-scoring one.

**Definition.** The **maximum-likelihood estimate** (MLE) of a parameter of interest $\theta$ is the value that maximises the
likelihood:

$$\hat\theta_{\text{MLE}} = \operatorname*{arg\,max}_{\theta\in\Theta}\ p(\mathcal D\mid\theta) = \operatorname*{arg\,max}_{\theta\in\Theta}\ \log p(\mathcal D\mid\theta) .$$

*In words:* among all allowed parameter values, pick the one under which the observed data were most probable. ("arg max"
returns the *location* of the maximum, not its height.) It is the most frequently used method for parameter estimation.

**Finding the maximum — the calculus recipe** (Module 14). Different problems call for different methods; when the
(log-)likelihood is differentiable:

1. Write down the (log-)likelihood of the data.
2. Take its derivative with respect to $\theta$ and find the **critical points**, where $\frac{d}{d\theta}\,p(\mathcal D\mid\theta) = 0$
   (equivalently $\frac{d}{d\theta}\log p(\mathcal D\mid\theta) = 0$, since $\frac{d}{d\theta}\log L = L'/L$ and $L > 0$).
3. **Check that the critical point is a maximum**, not a minimum or a saddle: e.g. the second derivative is negative there,
   or the first derivative changes sign from + to −.
4. **Check the boundary.** Sometimes the derivative is never 0 inside $\Theta$. Then $L$ is monotone there and the maximum
   is at an **endpoint (boundary) of the allowed range** $\Theta$ — the largest allowed value if $L$ increases, the
   smallest if it decreases. (So "if there is no critical point, use the maximum allowable parameter value" is only half
   the story: look at which way the likelihood slopes.)

### 4.1 Example 1: estimating a parameter

**Step 1 — Experiment.** Flip the new coin $n = 10$ times and collect the sample $\mathcal D = 1,1,1,0,0,0,0,0,1,0$
(1 = H, 0 = T). **Step 2 — Model.** Pick an appropriate model based on knowledge and assumptions: $X_i \sim
\text{Bernoulli}(p)$ indicates whether the $i$-th flip is 1, i.e. $f(x\mid\theta) = p^x(1-p)^{1-x}$ with $\theta = \{p\}$.
**Step 3 — Inference.** Estimate the parameter of interest $p$ from the data.

Many texts write the likelihood with a binomial coefficient, "$f(\mathcal D\mid p) = f(4\text{ heads}\mid p) = \binom{10}{4}p^4(1-p)^6$".

> ⚠️ **Common confusion — sequence vs count.** The probability of the *specific sequence* 1,1,1,0,0,0,0,0,1,0 is
> $p^4(1-p)^6$. The probability of the *event* "4 heads in 10 flips" (in any order) is $\binom{10}{4}p^4(1-p)^6$, because 210
> sequences have 4 heads. The two differ by the constant $\binom{10}{4} = 210$, which does not depend on $p$ — so both have
> exactly the same maximiser. Use whichever you like, but know which one you wrote.

**By calculus, without logs** (product rule):

$$\frac{d}{dp}\Big[\tbinom{10}{4}p^4(1-p)^6\Big] = \tbinom{10}{4}\Big[4p^3(1-p)^6 - 6p^4(1-p)^5\Big] = 0 .$$

*In words:* differentiate $p^4$ (giving $4p^3$) times $(1-p)^6$, plus $p^4$ times the derivative of $(1-p)^6$ (which is
$6(1-p)^5\cdot(-1)$). A sequence of algebraic steps finishes the job: for $0 < p < 1$,

$$4p^3(1-p)^6 = 6p^4(1-p)^5 \;\overset{\div\, p^3(1-p)^5}{\Longrightarrow}\; 4(1-p) = 6p \;\Longrightarrow\; 4 - 4p = 6p \;\Longrightarrow\; 4 = 10p \;\Longrightarrow\; \hat p = \frac{4}{10} .$$

*In words:* divide both sides by the common factor $p^3(1-p)^5$ (allowed because it is not 0 inside the interval), then
solve a linear equation. The binomial coefficient, a positive constant, plays no role in where the zero is.

**By calculus, with logs** (much easier):

$$\ell(p) = \log\tbinom{10}{4} + 4\log p + 6\log(1-p), \qquad \ell'(p) = \frac4p - \frac{6}{1-p} = 0
\;\Longrightarrow\; 4(1-p) = 6p \;\Longrightarrow\; \hat p = 0.4 .$$

*In words:* the log turns the product into a sum, each term differentiates on its own, and the constant $\log\binom{10}{4}$
disappears. Second derivative: $\ell''(p) = -4/p^2 - 6/(1-p)^2 < 0$ everywhere, so $\hat p = 0.4$ is the (only) maximum.
SymPy can check every step.

In [ ]:
# ── 4.1 Example 1 with SymPy: derivative, critical points, second derivative ──
p = sp.symbols("p", positive=True)
L_sym = sp.binomial(10, 4) * p ** 4 * (1 - p) ** 6
dL = sp.diff(L_sym, p)
print("L(p)   =", L_sym)
print("L'(p)  =", sp.factor(dL))
print("critical points of L:", sp.solve(sp.Eq(dL, 0), p), "(p = 0 and p = 1 are minima: L = 0 there)")
ell = sp.expand_log(sp.log(L_sym), force=True)
print("ℓ(p)   =", ell)
print("ℓ'(p)  =", sp.simplify(sp.diff(ell, p)), "→ solution", sp.solve(sp.diff(ell, p), p))
print("ℓ''(2/5) =", sp.diff(ell, p, 2).subs(p, sp.Rational(2, 5)), "< 0  → a maximum")
print("grid search on [0, 1]:", alpha_grid[np.argmax(L1)].round(4))

🔍 **What just happened?** SymPy factors the derivative as $420\,p^3(p-1)^5(5p - 2)$ (the same as
$210\,p^3(1-p)^5(4 - 10p)$), whose zeros are $p = 0$ (not listed, because we told SymPy that $p > 0$), $p = 2/5$ and $p = 1$;
the likelihood is 0 at both ends, so they are minima, and the maximum is at $p = 2/5 = 0.4$. The log-likelihood version has a
single critical point, $2/5$, with $\ell''(2/5) = -4/0.16 - 6/0.36 = -125/3 < 0$. A brute-force grid search agrees. So the MLE
is the observed fraction of heads, $\hat p = 4/10$.

### 4.2 "Thus the coin is not fair"? — not so fast

It is tempting to conclude "$\hat p = 0.4 \ne 0.5$, so given the sample it looks like the coin is not fair". The MLE alone
cannot support that conclusion: an estimate always differs a little from the truth (Module 22). How much better does
$p = 0.4$ explain the data than $p = 0.5$, and how often would a *fair* coin produce an MLE of 0.4 or less?

In [ ]:
# ── 4.2 Does the MLE 0.4 show that the coin is unfair? ──
ratio = bernoulli_likelihood(0.4, D1) / bernoulli_likelihood(0.5, D1)
p_low = stats.binom.cdf(4, 10, 0.5)
rng_f = np.random.default_rng(SEED + 42)
heads_fair = rng_f.binomial(10, 0.5, 100_000)                   # 100,000 ten-flip experiments with a FAIR coin
mle_fair = heads_fair / 10                                       # their MLEs
print(f"likelihood ratio L(0.4)/L(0.5) = {ratio:.3f}")
print(f"P(MLE ≤ 0.4 | fair coin) = P(≤ 4 heads in 10) = {p_low:.4f};  simulation: {np.mean(heads_fair <= 4):.4f}")
print(f"P(MLE at least as far from 0.5 as 0.4 is | fair coin) = {np.mean(np.abs(heads_fair - 5) >= 1):.4f} "
      f"(exact {1 - stats.binom.pmf(5, 10, 0.5):.4f})")

🔍 **What just happened?** $p = 0.4$ makes the data only 1.22 times more probable than $p = 0.5$, and a perfectly fair coin
gives an MLE of 0.4 or less in 37.7 % of ten-flip experiments (and an MLE at least 0.1 away from 0.5 in 75 % of them). Ten
flips cannot distinguish 0.4 from 0.5. The MLE is the *best single guess*, not evidence against fairness; measuring evidence
is the job of hypothesis tests (Module 25) and of the standard errors of Section 9.

### 4.3 When the maximum is on the boundary

Two situations where the derivative is never zero inside $\Theta$:

* **All observations equal.** Ten isolates, all resistant: $L(\alpha) = \alpha^{10}$, with $L'(\alpha) = 10\alpha^9 > 0$ on
  $(0, 1)$. The likelihood keeps increasing, so the maximum is at the right endpoint of $\Theta = [0, 1]$: $\hat\alpha = 1$.
  (Module 24 explains why a biologist may not like this answer, and how a prior fixes it.)
* **A parameter that bounds the data.** Molecules of a protein are scattered uniformly along a rod-shaped bacterium of
  unknown length $\theta$: positions $X_i\sim\text{Uniform}(0, \theta)$, density $1/\theta$ on $[0, \theta]$. Then
  $L(\theta) = \theta^{-n}$ if $\theta \ge \max_i x_i$ (every position must fit inside the cell) and $L(\theta) = 0$
  otherwise. On the allowed range $\theta \ge \max_i x_i$, $L'(\theta) = -n\theta^{-n-1} < 0$: the likelihood only
  decreases, so the maximum is at the left endpoint, $\hat\theta = \max_i x_i$.

In [ ]:
# ── 4.3 A boundary maximum: cell length from the positions of 8 SIMULATED fluorescent foci (true length 3.0 µm) ──
rng_b = np.random.default_rng(SEED + 43)
positions = rng_b.uniform(0, 3.0, 8)
theta_grid = np.linspace(0.5, 6, 1101)
L_unif = np.where(theta_grid >= positions.max(), theta_grid ** (-len(positions)), 0.0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(np.linspace(0, 1, 200), np.linspace(0, 1, 200) ** 10, color="tab:blue")
axes[0].plot(1, 1, "o", color="tab:red"); axes[0].set_xlabel("α"); axes[0].set_ylabel("L(α) = α¹⁰")
axes[0].set_title(r"10 of 10 isolates resistant: maximum at the boundary $\hat\alpha = 1$")
axes[1].plot(theta_grid, L_unif, color="tab:blue")
axes[1].plot(positions, np.zeros_like(positions), "|", color="k", ms=12, label="foci positions")
axes[1].axvline(positions.max(), color="tab:red", ls="--", label=rf"$\hat\theta$ = max position = {positions.max():.2f}")
axes[1].set_xlabel("cell length θ (µm)"); axes[1].set_ylabel("L(θ)"); axes[1].legend()
axes[1].set_title("Uniform(0, θ): L jumps up at max xᵢ, then only decreases")
plt.tight_layout(); plt.show()
print("positions (µm):", np.sort(positions).round(2), "| MLE θ̂ =", positions.max().round(3), "| true length 3.0")

🔍 **What just happened?** Left: $\alpha^{10}$ rises all the way to the edge of $[0, 1]$, so $\hat\alpha = 1$ although no
derivative is zero. Right: the uniform likelihood is 0 for $\theta$ below the largest observed position (a cell cannot be
shorter than the farthest molecule), jumps up at $\max_i x_i =$ 2.52 µm, and decreases afterwards; the MLE is that jump,
slightly below the true 3.0 µm. It *must* underestimate: the farthest molecule is never beyond the cell's end. (An MLE can be
biased — Section 6 shows another case.)

✏️ **Your turn.** (a) Find the MLE for 7 heads in 20 flips by hand, using the log recipe, and check with the grid.
(b) For the uniform model, show that $\hat\theta\cdot\frac{n+1}{n}$ is unbiased by simulating 10,000 cells with $n = 8$.

### 📘 Lecture notes — maximum likelihood and Example 1

**Definition.** $\hat\theta_{\text{MLE}} = \arg\max_{\theta\in\Theta} p(\mathcal D\mid\theta)$ — the most commonly used method for
parametric estimation. Equivalent: maximise $\log p(\mathcal D\mid\theta)$.

**Recipe.** (1) Write down the (log-)likelihood; (2) solve $\frac{d}{d\theta}\,p(\mathcal D\mid\theta) = 0$ (or the log
version); (3) check the critical point is a **maximum** (not a minimum or saddle); (4) if the derivative is never 0, the MLE is
at an **endpoint of the allowable range** — the end towards which the likelihood increases (all heads → $\hat\alpha = 1$;
Uniform$(0,\theta)$ → $\hat\theta = \max_i x_i$).

**Example 1.** $\mathcal D = 1,1,1,0,0,0,0,0,1,0$, $X_i\sim$ Bernoulli($p$), $f(x\mid\theta) = p^x(1-p)^{1-x}$.
$P(\mathcal D\mid h) = \binom{10}{4}p^4(1-p)^6$ (the probability of "4 heads"; the specific sequence has $p^4(1-p)^6$ — same
maximiser). $\frac{d}{dp}: \binom{10}{4}[4p^3(1-p)^6 - 6p^4(1-p)^5] = 0 \Rightarrow 4(1-p) = 6p \Rightarrow 4 = 10p
\Rightarrow \hat p = 4/10$. The binomial coefficient is a constant and does not affect the maximiser.

**Caution.** $\hat p = 0.4$ does *not* show the coin is unfair: $L(0.4)/L(0.5) = 1.22$, and a fair coin gives $\hat p \le 0.4$ in
37.7 % of ten-flip experiments.

> 🎓 **Exam tip.** Always say *why* you may divide by $p^3(1-p)^5$ (it is non-zero for $0 < p < 1$) and always show the
> maximum check — a second derivative or a sign change. Graders look for both.

---
## 5 · The Bernoulli MLE in general

> 📍 **Where we are.** Example 1 had 4 successes in 10 trials. The same steps work for any $s$ successes in $n$ trials, and
> give the formula every biologist uses without thinking: "the fraction of successes".

**Scenario.** We flip a coin $n$ times but do not know whether it is fair. Each trial $X_i$ is independent and follows a
Bernoulli distribution with unknown parameter $\alpha$: $X_i\sim\text{Bernoulli}(\alpha)$, so
$f(X_i = x\mid\theta) = \alpha^x(1-\alpha)^{1-x}$ with $\theta = \{\alpha\}$. Combining the trials gives something that
looks like the binomial distribution: if $k$ of the $n$ flips were heads, $p(\mathcal D\mid\theta) = \binom nk\alpha^k(1-\alpha)^{n-k}$
(again, the coefficient is optional).

**Step 1 — write down the log-likelihood.** With $s = \sum_i x_i$ successes,

$$\log p(\mathcal D\mid\theta) = \log\prod_{i=1}^{n} p(X_i\mid\theta) = \log\big[\alpha^{s}(1-\alpha)^{n-s}\big] = s\log\alpha + (n-s)\log(1-\alpha) .$$

*In words:* the product of $n$ Bernoulli factors collects into $\alpha^s(1-\alpha)^{n-s}$, and the log rules $\log(ab) =
\log a + \log b$ and $\log a^k = k\log a$ split it into two terms.

**Step 2 — maximise.** Differentiate term by term, using $\frac{d}{d\alpha}\log\alpha = \frac1\alpha$ and, by the chain rule,
$\frac{d}{d\alpha}\log(1-\alpha) = \frac{1}{1-\alpha}\cdot(-1)$:

$$\frac{d}{d\alpha}\big[s\log\alpha + (n-s)\log(1-\alpha)\big] = \frac{s}{\alpha} - \frac{n-s}{1-\alpha} = 0
\;\Longrightarrow\; s(1-\alpha) = (n-s)\alpha \;\Longrightarrow\; s - s\alpha = n\alpha - s\alpha \;\Longrightarrow\; \hat\alpha = \frac sn .$$

*In words:* set the derivative to zero, multiply both sides by $\alpha(1-\alpha)$, and the $s\alpha$ terms cancel. The
second derivative, $-s/\alpha^2 - (n-s)/(1-\alpha)^2$, is negative, so it is a maximum (if $0 < s < n$; if $s = 0$ or $s = n$
the maximum is on the boundary, $\hat\alpha = 0$ or $1$, Section 4.3). For a set of $n$ Bernoulli random variables with $s$
successes the MLE is the **observed fraction of successes** — which aligns with Example 1 ($4/10$).

In [ ]:
# ── 5.1 The general Bernoulli MLE with SymPy (symbols s, n) ──
a, s, n = sp.symbols("alpha s n", positive=True)
loglik = s * sp.log(a) + (n - s) * sp.log(1 - a)
d1 = sp.diff(loglik, a)
print("dℓ/dα   =", d1)
print("solution:", sp.solve(sp.Eq(d1, 0), a))
print("d²ℓ/dα² =", sp.simplify(sp.diff(loglik, a, 2)))
print("Example 1 (s = 4, n = 10):", sp.solve(sp.Eq(d1.subs({s: 4, n: 10}), 0), a))

🔍 **What just happened?** SymPy reproduces the derivative $s/\alpha - (n-s)/(1-\alpha)$ (it writes the second term as
$-(n-s)/(1-\alpha)$ in its own order) and solves it to $\hat\alpha = s/n$; with $s = 4$, $n = 10$ it returns $2/5$. The second
derivative is a sum of two negative terms.

### 5.2 Allele frequency from genotypes

A diploid person's genotype $g \in\{0, 1, 2\}$ counts copies of an allele. Under Hardy–Weinberg equilibrium each of the two
copies is the allele with probability $p$, independently, so $g\sim\text{Binomial}(2, p)$:
$P(g\mid p) = \binom{2}{g}p^g(1-p)^{2-g}$. For $n$ people with $n_0, n_1, n_2$ individuals of genotypes 0, 1, 2:

$$L(p) = \prod_{i=1}^n\binom{2}{g_i}p^{g_i}(1-p)^{2-g_i} = C\cdot p^{\sum_i g_i}(1-p)^{2n - \sum_i g_i}, \qquad \sum_i g_i = n_1 + 2n_2 ,$$

*In words:* every person contributes two "coin flips" (their two allele copies); the constant $C = \prod_i\binom{2}{g_i} =
2^{n_1}$ does not involve $p$. This is the Bernoulli likelihood with $s = n_1 + 2n_2$ successes out of $2n$ trials, so

$$\hat p = \frac{2n_2 + n_1}{2n} .$$

*In words:* count the allele copies (two in each homozygote of type 2, one in each heterozygote) and divide by the number of
chromosomes — "allele counting" is maximum likelihood.

In [ ]:
# ── 5.2 Allele-frequency MLE from 500 SIMULATED genotypes (true p = 0.3) ──
n0, n1, n2 = np.bincount(genotypes, minlength=3)
n_people = len(genotypes)
p_hat = (2 * n2 + n1) / (2 * n_people)
p_grid = np.linspace(0.001, 0.999, 9981)
ll_p = (n1 + 2 * n2) * np.log(p_grid) + (2 * n_people - n1 - 2 * n2) * np.log(1 - p_grid)
n0, n1, n2 = int(n0), int(n1), int(n2)
print(f"n0, n1, n2 = {n0}, {n1}, {n2};  p̂ = (2·{n2} + {n1}) / (2·{n_people}) = {p_hat:.4f};  grid search: {p_grid[ll_p.argmax()]:.4f}")
print(f"standard error (Module 22): √(p̂(1 − p̂)/2n) = {np.sqrt(p_hat * (1 - p_hat) / (2 * n_people)):.4f}")
expected = n_people * np.array([(1 - p_hat) ** 2, 2 * p_hat * (1 - p_hat), p_hat ** 2])
print("observed genotype counts:", [n0, n1, n2], "| expected under Hardy–Weinberg from p̂:", expected.round(1).tolist())

🔍 **What just happened?** The 500 simulated people have genotype counts $n_0, n_1, n_2 = 231, 226, 43$, so
$\hat p = (2\cdot 43 + 226)/1000 = 0.312$ — allele counting and the grid search agree. The truth is 0.3: the estimate is
off by 0.012, less than one standard error (0.0147), exactly the size of error Module 22 taught us to expect. The genotype
counts expected under Hardy–Weinberg from $\hat p$ (236.7, 214.7, 48.7) are close to the observed ones (a few more
heterozygotes than expected, by chance); Module 26 turns that comparison into a test.

### 🌉 Bridge — allele frequencies, genotype likelihoods and GWAS

Allele counting is the MLE only when genotypes are known exactly. In low-coverage sequencing they are not: each person
contributes a **genotype likelihood** $P(\text{reads}\mid g)$ for $g = 0, 1, 2$, and the population allele frequency is found by
maximising $\prod_i\sum_g P(\text{reads}_i\mid g)P(g\mid p)$ — no closed form, so it is done numerically (Section 8) or by
the EM algorithm ([BiologyToML 13 · mixture models and EM](../../../../ml/tutorial/BiologyToML/notebooks/13_mixture_models_em.ipynb)). Allele-frequency estimates
feed Hardy–Weinberg tests (Module 26), association tests and population-structure analyses
([BiologyToML 35 · statistical genetics](../../../../ml/tutorial/BiologyToML/notebooks/35_statistical_genetics.ipynb)).

✏️ **Your turn.** Without Hardy–Weinberg, the three genotype frequencies $\pi_0, \pi_1, \pi_2$ form a categorical model. By
analogy with $s/n$, what are their MLEs? Compute them for the 500 people and compare with the HWE predictions above.

---
## 6 · The normal (Gaussian) MLE

> 📍 **Where we are.** Continuous measurements — log-expression, cholesterol, height — are usually modelled as normal. Now
> $\theta = \{\mu, \sigma^2\}$ has two components, so we take two (partial) derivatives.

Let $\mathcal D = X_1,\dots,X_n \sim \mathcal N(\mu, \sigma^2)$ i.i.d., with observed values $x_1, \dots, x_n$. Find the MLE of
$\theta = \{\mu, \sigma^2\}$.

**Step 1 — write down the log-likelihood.** The normal density is $p(x\mid\mu,\sigma^2) = \frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{(x-\mu)^2}{2\sigma^2}}$. Multiply $n$ of them:

$$p(\mathcal D\mid\mu,\sigma^2) = \prod_{i=1}^n p(x_i\mid\mu,\sigma^2) = \prod_{i=1}^n\frac{1}{\sigma\sqrt{2\pi}}\,e^{-\frac{(x_i-\mu)^2}{2\sigma^2}}
= \Big(\frac{1}{\sigma\sqrt{2\pi}}\Big)^{n} e^{-\frac{1}{2\sigma^2}\sum_{i=1}^n(x_i-\mu)^2} .$$

*In words:* the $n$ identical constants multiply to the $n$-th power, and $e^{a_1}e^{a_2}\cdots e^{a_n} = e^{a_1 + \dots + a_n}$
collects the exponents into one sum. Note that each term has its own $x_i$ — the sum runs over the data.

Now take logs, using $\log(e^y) = y$, $\log a^n = n\log a$ and $\log\frac{1}{ab} = -\log a - \log b$:

$$\log p(\mathcal D\mid\mu,\sigma^2) = \log\Big(\frac{1}{\sigma\sqrt{2\pi}}\Big)^{n} - \frac{1}{2\sigma^2}\sum_{i=1}^n(x_i-\mu)^2
= -n\log\sqrt{2\pi} - n\log\sigma - \frac{1}{2\sigma^2}\sum_{i=1}^n(x_i-\mu)^2 .$$

*In words:* a constant, a term that penalises large $\sigma$, and a term that penalises squared distances between the data and
$\mu$ — measured in units of the variance.

**Step 2a — maximise with respect to $\mu$.** The first two terms do not contain $\mu$, so their derivative is 0. For the
third, the chain rule gives $\frac{d}{d\mu}(x_i - \mu)^2 = 2(x_i - \mu)\cdot(-1)$:

$$\frac{\partial}{\partial\mu}\log p = -\frac{1}{2\sigma^2}\sum_{i=1}^n 2(x_i-\mu)(-1) = \frac{1}{\sigma^2}\sum_{i=1}^n(x_i - \mu) = 0
\;\Longrightarrow\; \sum_{i=1}^n x_i - n\mu = 0 \;\Longrightarrow\; \hat\mu_{\text{MLE}} = \frac1n\sum_{i=1}^n x_i = \bar x .$$

*In words:* the MLE of the mean is the **sample mean**: the value that balances the positive and negative deviations.

**Step 2b — maximise with respect to $\sigma$.** Now $-n\log\sqrt{2\pi}$ is the constant (derivative 0). Write
$\frac{1}{2\sigma^2} = \frac12\sigma^{-2}$, whose derivative is $\frac12(-2)\sigma^{-3} = -\sigma^{-3}$:

$$\frac{\partial}{\partial\sigma}\log p = 0 - \frac{n}{\sigma} - \frac12(-2)\sigma^{-3}\sum_{i=1}^n(x_i-\mu)^2 = -\frac n\sigma + \sigma^{-3}\sum_{i=1}^n(x_i-\mu)^2 = 0 .$$

*In words:* the first term pulls $\sigma$ down (narrow curves are taller), the second pushes it up (wide curves forgive large
deviations). Solve, one step at a time:

$$\sigma^{-3}\sum_{i=1}^n(x_i-\mu)^2 = \frac n\sigma \;\overset{\times\,\sigma^3}{\Longrightarrow}\; \sum_{i=1}^n(x_i-\mu)^2 = n\sigma^2
\;\Longrightarrow\; \hat\sigma^2_{\text{MLE}} = \frac1n\sum_{i=1}^n(x_i - \hat\mu_{\text{MLE}})^2 .$$

*In words:* the MLE of the variance is the **average squared deviation from the sample mean**, dividing by $n$ (not
$n - 1$). We plugged in $\hat\mu$ because both equations must hold at once, and the $\mu$-equation gives $\bar x$ whatever
$\sigma$ is. Differentiating with respect to $\sigma^2$ instead of $\sigma$ gives the same answer (Section 9 explains why).

In [ ]:
# ── 6.1 Check the normal MLE on 6 SIMULATED expression values: formulas, SymPy, and the σ² parametrisation ──
x6 = np.array([7.1, 8.4, 7.9, 8.8, 7.6, 8.2])                 # log2 expression of a gene in six mice (SIMULATED)
mu_s, sig_s, v_s = sp.symbols("mu sigma v", positive=True)
xs_sym = [sp.Rational(str(v)) for v in x6]
loglik_sigma = sum(-sp.log(sp.sqrt(2 * sp.pi)) - sp.log(sig_s) - (xi - mu_s) ** 2 / (2 * sig_s ** 2) for xi in xs_sym)
sol = sp.solve([sp.diff(loglik_sigma, mu_s), sp.diff(loglik_sigma, sig_s)], [mu_s, sig_s], dict=True)[0]
loglik_var = sum(-sp.log(sp.sqrt(2 * sp.pi * v_s)) - (xi - mu_s) ** 2 / (2 * v_s) for xi in xs_sym)
sol_v = sp.solve([sp.diff(loglik_var, mu_s), sp.diff(loglik_var, v_s)], [mu_s, v_s], dict=True)[0]
print(f"formulas : μ̂ = x̄ = {x6.mean():.4f},  σ̂² = mean((x − x̄)²) = {np.mean((x6 - x6.mean()) ** 2):.4f}")
print(f"SymPy (θ = μ, σ) : μ̂ = {sol[mu_s]} = {float(sol[mu_s]):.4f},  σ̂² = {float(sol[sig_s] ** 2):.4f}")
print(f"SymPy (θ = μ, σ²): μ̂ = {float(sol_v[mu_s]):.4f},  σ̂² = {sol_v[v_s]} = {float(sol_v[v_s]):.4f}")
print(f"np.var(x) = {np.var(x6):.4f} (ddof = 0, the MLE)   np.var(x, ddof=1) = {np.var(x6, ddof=1):.4f} (the n − 1 version)")

🔍 **What just happened?** For the six values the formulas give $\hat\mu = 8.0$ and $\hat\sigma^2 =$ 0.3033. SymPy solves the
two derivative equations exactly ($\hat\mu = 8$) and gets the same variance whether the second parameter is $\sigma$ or
$\sigma^2$. NumPy's default `np.var` (`ddof=0`) *is* the MLE; `ddof=1` divides by $n - 1 = 5$ instead and gives 0.3640.
Let us look at the whole likelihood surface.

In [ ]:
# ── 6.2 The log-likelihood surface ℓ(μ, σ) of the six values ──
mu_g, sd_g = np.meshgrid(np.linspace(7.0, 9.0, 201), np.linspace(0.2, 1.4, 201))
ll_surface = sum(stats.norm.logpdf(xi, mu_g, sd_g) for xi in x6)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
cs = ax.contourf(mu_g, sd_g, ll_surface, levels=np.linspace(ll_surface.max() - 12, ll_surface.max(), 13), cmap="viridis")
plt.colorbar(cs, ax=ax, label="log-likelihood ℓ(μ, σ)")
mu_hat6, sd_hat6 = x6.mean(), x6.std()
ax.plot(mu_hat6, sd_hat6, "r*", ms=14, label=rf"MLE $(\hat\mu, \hat\sigma)$ = ({mu_hat6:.2f}, {sd_hat6:.3f})")
ax.set_xlabel("μ (log2 expression)"); ax.set_ylabel("σ"); ax.legend(loc="upper right")
ax.set_title("SIMULATED six mice: the log-likelihood is a single hill")
plt.tight_layout(); plt.show()
i_best = np.unravel_index(ll_surface.argmax(), ll_surface.shape)
print(f"grid maximum at μ = {mu_g[i_best]:.3f}, σ = {sd_g[i_best]:.3f};  ℓ_max = {ll_surface.max():.3f}")

🔍 **What just happened?** The log-likelihood over the $(\mu, \sigma)$ plane is a single hill, and its top (red star, grid
maximum at $\mu = 8.000$, $\sigma = 0.548$, the grid point nearest to $\hat\sigma = \sqrt{0.3033} = 0.551$) is at $(\bar x, \hat\sigma)$. The hill is narrow in the $\mu$ direction and asymmetric in $\sigma$:
values of $\sigma$ much smaller than the spread of the data are punished hard (the data would be "impossible"), larger ones
more gently.

### 6.3 The MLE of the variance is biased

Dividing by $n$ makes $\hat\sigma^2$ too small on average. Here is the proof, one line at a time. First an algebraic identity:
write $x_i - \mu = (x_i - \bar x) + (\bar x - \mu)$ and expand the square,

$$\sum_{i=1}^n(x_i-\mu)^2 = \sum_{i=1}^n(x_i-\bar x)^2 + 2(\bar x - \mu)\underbrace{\sum_{i=1}^n(x_i - \bar x)}_{=\,0} + n(\bar x-\mu)^2
\;\Longrightarrow\; \sum_{i=1}^n(x_i-\bar x)^2 = \sum_{i=1}^n(x_i-\mu)^2 - n(\bar x-\mu)^2 .$$

*In words:* squared deviations from the sample mean equal squared deviations from the true mean minus a correction for how far
the sample mean itself is from the truth (the middle term vanishes because deviations from $\bar x$ sum to zero). Now take
expectations, using $E[(X_i-\mu)^2] = \sigma^2$ and $E[(\bar X-\mu)^2] = \operatorname{Var}(\bar X) = \sigma^2/n$ (Module 22):

$$E\Big[\sum_{i=1}^n(X_i-\bar X)^2\Big] = n\sigma^2 - n\cdot\frac{\sigma^2}{n} = (n-1)\sigma^2
\;\Longrightarrow\; E[\hat\sigma^2_{\text{MLE}}] = \frac{n-1}{n}\sigma^2, \qquad E\Big[\frac{1}{n-1}\sum_{i=1}^n(X_i-\bar X)^2\Big] = \sigma^2 .$$

*In words:* the MLE underestimates the variance by the factor $(n-1)/n$, because the data are closer to their own mean than
to the true one. Dividing by $n - 1$ instead (Bessel's correction, Module 21) removes the bias. The bias vanishes as
$n\to\infty$.

In [ ]:
# ── 6.3 Bias of σ̂²_MLE by simulation: 100,000 SIMULATED experiments per sample size, true σ² = 0.36 ──
rng_bias = np.random.default_rng(SEED + 63)
rows = []
for n_ in [2, 5, 10, 50]:
    x = rng_bias.normal(8.0, 0.6, (100_000, n_))
    rows.append({"n": n_, "mean of σ̂²_MLE (÷n)": x.var(axis=1).mean(), "(n−1)/n · σ²": (n_ - 1) / n_ * 0.36,
                 "mean of s² (÷(n−1))": x.var(axis=1, ddof=1).mean(), "σ²": 0.36})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

🔍 **What just happened?** With $n = 5$ mice the MLE of the variance averages 0.2879 instead of 0.36 — exactly
$(4/5)\times 0.36 = 0.288$ — while $s^2$ averages 0.3599. With 2 mice the MLE averages half the truth; with 50 the bias is 2 %.
The MLE is not always unbiased; what it *does* guarantee is consistency (Section 9). In practice: report $s^2$ (or $s$) for
small samples, and remember that `np.var` defaults to the MLE.

✏️ **Your turn.** (a) By hand: for the expression values 2, 4, 6 compute $\hat\mu$, $\hat\sigma^2_{\text{MLE}}$ and $s^2$.
(b) Is $\hat\sigma = \sqrt{\hat\sigma^2}$ unbiased for $\sigma$ even when you divide by $n - 1$? Check by simulation (Module 22,
Section 6.1 has the answer for $n = 20$).

---
## 7 · The Poisson MLE — and two relatives

> 📍 **Where we are.** Counts — molecules per cell, mutations per genome, reads per gene — are the bread and butter of
> genomics. Their simplest model is the Poisson distribution (Module 18).

Let $\mathcal D = X_1,\dots,X_n\sim\text{Poisson}(\lambda)$ i.i.d. Find the MLE of $\theta = \{\lambda\}$.

**Step 1 — write down the log-likelihood.** With $P(x\mid\lambda) = e^{-\lambda}\lambda^x/x!$,

$$P(\mathcal D\mid\lambda) = \prod_{i=1}^n e^{-\lambda}\frac{\lambda^{x_i}}{x_i!} = e^{-n\lambda}\,\frac{\lambda^{\sum_{i=1}^n x_i}}{\prod_{i=1}^n x_i!}, \qquad
\log P(\mathcal D\mid\lambda) = -n\lambda + \Big(\sum_{i=1}^n x_i\Big)\log\lambda - \sum_{i=1}^n\log(x_i!) .$$

*In words:* $n$ factors $e^{-\lambda}$ multiply to $e^{-n\lambda}$, the powers of $\lambda$ add their exponents, and the
factorials collect in the denominator; the log turns these into a sum of three terms, the last of which does not involve
$\lambda$.

**Step 2 — maximise.** The term $-\sum_i\log(x_i!)$ is a constant (derivative 0):

$$\frac{d}{d\lambda}\log P(\mathcal D\mid\lambda) = -n + \frac1\lambda\sum_{i=1}^n x_i = 0 \;\Longrightarrow\; \sum_{i=1}^n x_i = n\lambda
\;\Longrightarrow\; \hat\lambda_{\text{MLE}} = \frac1n\sum_{i=1}^n x_i = \bar x .$$

*In words:* the MLE of a Poisson rate is the **average count**. The second derivative, $-\sum_i x_i/\lambda^2$, is negative
whenever at least one count is positive (if all counts are 0, the maximum is on the boundary, $\hat\lambda = 0$).

### 7.1 Biology: the expression rate of each marker gene in each cell type

In [ ]:
# ── 7.1 Poisson MLE λ̂ = x̄ for every marker gene in every cell type, vs the simulator's true rates ──
cell_types = list(CELL_TYPE_MARKERS)
markers = [g for t in cell_types for g in CELL_TYPE_MARKERS[t]]
lam_hat = pd.DataFrame({g: [counts[labels == t, genes.index(g)].mean() for t in cell_types] for g in markers}, index=cell_types)
print("λ̂ = mean count per cell (rows: cell type; columns: marker gene):")
print(lam_hat.round(2).to_string())
err = (lam_hat - true_rates[markers]).to_numpy()
n_per_type = np.array([(labels == t).sum() for t in cell_types])[:, None]
se = np.sqrt(lam_hat.to_numpy() / n_per_type)                       # SE of a Poisson mean: √(λ/n)
print(f"\nlargest |λ̂ − λ_true| = {np.abs(err).max():.3f};  fraction of the 48 estimates within 2 SE of the truth: "
      f"{np.mean(np.abs(err) <= 2 * se):.3f}")

🔍 **What just happened?** Each entry is the MLE of one rate: e.g. *CD3E* in T cells is estimated at 6.61 molecules
per cell (truth 6.71), *LYZ* in macrophages at 12.63 (truth 12.70), and the off-type rates are all below 0.6. The
largest error over all 48 estimates is 0.56 molecules per cell, and all 48 lie within two standard errors ($\sqrt{\hat\lambda/n}$)
of the truth — the normal approximation of Module 22 predicts about 95 % of them would, so this is a little lucky. A cell-type "signature" is just a table
of Poisson MLEs.

### 7.2 Rates with exposure: the mutation rate per base

Mutation-accumulation experiments propagate $m$ lines of an organism through bottlenecks for many generations, then
sequence them. Line $i$ has $k_i$ new mutations over an "exposure" $T_i$ = (callable bases) × (generations). If mutations
arise at rate $\mu$ per base per generation, $k_i\sim\text{Poisson}(\mu T_i)$. The log-likelihood and its derivative:

$$\ell(\mu) = \sum_{i=1}^m\big[k_i\log(\mu T_i) - \mu T_i - \log k_i!\big], \qquad \ell'(\mu) = \frac{\sum_i k_i}{\mu} - \sum_i T_i = 0
\;\Longrightarrow\; \hat\mu = \frac{\sum_i k_i}{\sum_i T_i} .$$

*In words:* total events divided by total exposure — not the average of the per-line rates $k_i/T_i$, which would give
equal weight to short and long lines.

In [ ]:
# ── 7.2 Mutation-rate MLE from 8 SIMULATED mutation-accumulation lines (illustrative rate 3e-10 per base per generation) ──
rng_mut = np.random.default_rng(SEED + 62)
mu_true = 3e-10
callable_bases = rng_mut.uniform(10.5e6, 11.5e6, 8)                   # a yeast-sized genome (≈ 12 Mb)
generations = rng_mut.choice([1000, 1500, 2000, 4000], 8)
exposure = callable_bases * generations
k_mut = rng_mut.poisson(mu_true * exposure)
mu_hat = k_mut.sum() / exposure.sum()
print("mutations per line:", k_mut.tolist(), "| generations:", generations.tolist())
print(f"MLE μ̂ = Σk / ΣT = {k_mut.sum()} / {exposure.sum():.3e} = {mu_hat:.3e} per base per generation  (truth {mu_true:.1e})")
print(f"average of per-line rates = {np.mean(k_mut / exposure):.3e} (not the MLE);  "
      f"SE ≈ √Σk / ΣT = {np.sqrt(k_mut.sum()) / exposure.sum():.1e}")

🔍 **What just happened?** Eight lines accumulated 3, 15, 5, 2, 7, 19, 6 and 2 mutations; the MLE pools them: $3.19\times10^{-10}$ per base per generation,
close to the illustrative truth $3\times10^{-10}$ (a value of the order reported for budding yeast). With only 59
mutations in total the relative standard error is about $1/\sqrt{\sum k}$ ≈ 13 % — mutation rates are expensive to
measure precisely.

### 7.3 Waiting times: how long does a transcription factor stay bound?

Single-molecule imaging records how long each fluorescently labelled transcription-factor molecule stays bound to DNA. For a
single-step unbinding process the **residence times** are exponential, $f(x\mid k) = k e^{-kx}$ ($x \ge 0$), where $k$ is the
off-rate. Then

$$\ell(k) = \sum_{i=1}^n\big[\log k - kx_i\big] = n\log k - k\sum_{i=1}^n x_i, \qquad \ell'(k) = \frac nk - \sum_{i=1}^n x_i = 0
\;\Longrightarrow\; \hat k = \frac{n}{\sum_i x_i} = \frac{1}{\bar x} .$$

*In words:* the MLE of a rate is one over the mean waiting time; $\ell''(k) = -n/k^2 < 0$ confirms the maximum.

In [ ]:
# ── 7.3 Exponential MLE for 200 SIMULATED residence times (illustrative off-rate 0.5 per second) ──
rng_tf = np.random.default_rng(SEED + 73)
k_true = 0.5
residence = rng_tf.exponential(1 / k_true, 200)
k_hat = 1 / residence.mean()
grid_t = np.linspace(0, residence.max(), 200)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(residence, bins=25, density=True, color="tab:orange", alpha=0.6, label="200 binding events")
ax.plot(grid_t, k_hat * np.exp(-k_hat * grid_t), "k-", label=rf"fitted exponential, $\hat k = 1/\bar x$ = {k_hat:.3f} s$^{{-1}}$")
ax.set_xlabel("residence time on DNA (s)"); ax.set_ylabel("density"); ax.legend()
ax.set_title("SIMULATED single-molecule imaging: exponential MLE")
plt.tight_layout(); plt.show()
print(f"mean residence time x̄ = {residence.mean():.3f} s → k̂ = {k_hat:.3f} per s (truth {k_true}); "
      f"half-life log 2 / k̂ = {np.log(2) / k_hat:.2f} s")

🔍 **What just happened?** The 200 simulated binding events last 1.923 s on average, so $\hat k = 1/\bar x =$ 0.520 per
second (truth 0.5), and the fitted exponential curve follows the histogram. The MLE of a waiting-time model is one line of
code once the derivation is done.

### 🌉 Bridge — count models in genomics

The Poisson MLE $\hat\lambda = \bar x$ is the starting point of every count model in genomics. RNA-seq tools model reads per
gene as **negative binomial** (Section 2's over-dispersion) and estimate a mean and a dispersion for each gene by maximum
likelihood — numerically, as in Section 8 — and generalised linear models let the mean depend on covariates
([BiologyToML 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb)). Single-cell methods fit the same models per gene and
per cell type ([BiologyToML 30 · single-cell RNA-seq](../../../../ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb)). Exposures (Section 7.2) appear as
**size factors** — sequencing depth per sample — that multiply the rate exactly like $T_i$.

✏️ **Your turn.** (a) By hand: counts 3, 0, 2, 5, 0 — what is $\hat\lambda$? (b) Show that for the exposure model with all
$T_i$ equal, $\hat\mu$ reduces to the Poisson $\bar x / T$.

### 📘 Lecture notes — the Bernoulli, normal and Poisson MLEs

**Two steps.** (1) Write down the log-likelihood of the data; (2) maximise it (usually by calculus).

| Model | log-likelihood | set derivative to 0 | MLE |
|---|---|---|---|
| Bernoulli($\alpha$), $s$ successes | $s\log\alpha + (n-s)\log(1-\alpha)$ | $\frac s\alpha - \frac{n-s}{1-\alpha} = 0$ | $\hat\alpha = s/n$ |
| Normal($\mu, \sigma^2$) | $-n\log\sqrt{2\pi} - n\log\sigma - \frac{1}{2\sigma^2}\sum_i(x_i - \mu)^2$ | $\frac{1}{\sigma^2}\sum_i(x_i - \mu) = 0$; $-\frac n\sigma + \sigma^{-3}\sum_i(x_i - \mu)^2 = 0$ | $\hat\mu = \bar x$, $\hat\sigma^2 = \frac1n\sum_i(x_i - \bar x)^2$ |
| Poisson($\lambda$) | $-n\lambda + (\sum_i x_i)\log\lambda - \sum_i\log x_i!$ | $-n + \frac1\lambda\sum_i x_i = 0$ | $\hat\lambda = \bar x$ |
| Exponential($k$) | $n\log k - k\sum_i x_i$ | $\frac nk - \sum_i x_i = 0$ | $\hat k = 1/\bar x$ |

*In words:* every one of these MLEs is a natural "sample version" of the parameter: a fraction, a mean, a mean squared deviation.

**Derivative tools.** $\frac{d}{d\alpha}\log(1-\alpha) = -\frac{1}{1-\alpha}$ (chain rule; in general $\frac{d}{dx}[a\log f(x)] = a\frac{f'(x)}{f(x)}$);
$\frac{d}{d\mu}(x_i-\mu)^2 = -2(x_i-\mu)$; $\frac{d}{d\sigma}\frac{1}{2\sigma^2} = -\sigma^{-3}$. Terms without the parameter
($n\log\sqrt{2\pi}$, $\sum\log x_i!$, $\log\binom nk$) have derivative 0.

**Bias.** $E[\hat\sigma^2_{\text{MLE}}] = \frac{n-1}{n}\sigma^2$ (proof via $\sum(x_i-\bar x)^2 = \sum(x_i-\mu)^2 - n(\bar x-\mu)^2$);
$s^2 = \frac{1}{n-1}\sum(x_i-\bar x)^2$ is unbiased.

**Worked examples.** Example 1: $\hat\alpha = 4/10$. Six mice: $\hat\mu = 8.0$, $\hat\sigma^2 =$ 0.3033. Allele counting:
$\hat p = (2n_2 + n_1)/2n =$ 0.312. *CD3E* in T cells: $\hat\lambda =$ 6.61.

> 🎓 **Exam tip.** In the normal derivation write $x_i$ (with the index!) inside every sum, cancel only factors that do not
> involve the parameter you are differentiating, and remember that the variance MLE divides by $n$.

---
## 8 · Maximum likelihood by computer

> 📍 **Where we are.** Calculus gave closed-form MLEs for textbook models. Real models — over-dispersion, dropout,
> mixtures, phylogenetic trees — usually have no closed form. Then we let an optimiser climb the log-likelihood for us.

**The numerical recipe.** Write a function that returns the **negative** log-likelihood (optimisers *minimise* by convention)
and hand it to `scipy.optimize.minimize`. Two practical tricks: (1) optimise parameters on an **unconstrained** scale —
$\log\lambda$ instead of $\lambda > 0$, $\operatorname{logit}\alpha = \log\frac{\alpha}{1-\alpha}$ instead of $0 < \alpha < 1$ —
so the optimiser cannot step outside $\Theta$; (2) always check a numerical answer against a closed form when one exists.

In [ ]:
# ── 8.1 Numerical MLE with scipy.optimize.minimize — checked against the closed forms ──
def expit(t):
    """Inverse logit: maps the real line to (0, 1)."""
    return 1 / (1 + np.exp(-t))


def nll_bernoulli(params, x):
    """Negative Bernoulli log-likelihood; params = [logit α]."""
    alpha = expit(params[0])
    return -np.sum(x * np.log(alpha) + (1 - x) * np.log(1 - alpha))


def nll_normal(params, x):
    """Negative normal log-likelihood; params = [μ, log σ]."""
    return -np.sum(stats.norm.logpdf(x, params[0], np.exp(params[1])))


def nll_poisson(params, x):
    """Negative Poisson log-likelihood; params = [log λ]."""
    return -np.sum(stats.poisson.logpmf(x, np.exp(params[0])))


fit_b = minimize(nll_bernoulli, x0=[0.0], args=(D1,))
fit_n = minimize(nll_normal, x0=[7.0, 0.0], args=(x6,))
fit_p = minimize(nll_poisson, x0=[0.0], args=(cd3e_t,))
print(f"Bernoulli, Example 1 : optimiser α̂ = {expit(fit_b.x[0]):.5f}   closed form s/n = {D1.mean():.5f}")
print(f"normal, six mice     : optimiser μ̂ = {fit_n.x[0]:.5f}, σ̂² = {np.exp(2 * fit_n.x[1]):.5f}   "
      f"closed form {x6.mean():.5f}, {np.var(x6):.5f}")
print(f"Poisson, CD3E/T cells: optimiser λ̂ = {np.exp(fit_p.x[0]):.5f}   closed form x̄ = {cd3e_t.mean():.5f}")

🔍 **What just happened?** The optimiser, which knows nothing about calculus formulas, lands on the closed-form answers to
four or five decimals in all three models: 0.4 for Example 1, $(8.0, $ 0.3033$)$ for the six mice, and 6.61 for
*CD3E*. That agreement is our licence to trust it on a model without a formula.

### 8.2 A model without a closed form: zero-inflated counts

In droplet-based single-cell RNA-seq, some transcripts present in a cell are simply not captured. A crude but instructive
model is the **zero-inflated Poisson** (ZIP): with probability $\pi$ the gene "drops out" and we observe 0; otherwise we
observe a Poisson($\lambda$) count. So

$$P(X = 0) = \pi + (1-\pi)e^{-\lambda}, \qquad P(X = k) = (1-\pi)\frac{e^{-\lambda}\lambda^k}{k!}\quad (k \ge 1) .$$

*In words:* a zero can come from two sources — a dropout, or a genuine Poisson zero — while a positive count can only come
from the Poisson part. Setting the two partial derivatives of the log-likelihood to zero gives equations that cannot be solved
in closed form, so we optimise numerically.

In [ ]:
# ── 8.2 Fit Poisson vs zero-inflated Poisson to 400 SIMULATED cells (true dropout π = 0.4, true rate λ = 3) ──
rng_zip = np.random.default_rng(SEED + 82)
pi_true, lam_true = 0.4, 3.0
zip_counts = np.where(rng_zip.random(400) < pi_true, 0, rng_zip.poisson(lam_true, 400))


def nll_zip(params, x):
    """Negative zero-inflated-Poisson log-likelihood; params = [logit π, log λ]."""
    pi, lam = expit(params[0]), np.exp(params[1])
    log_p0 = np.log(pi + (1 - pi) * np.exp(-lam))
    log_pk = np.log(1 - pi) + stats.poisson.logpmf(x, lam)
    return -np.sum(np.where(x == 0, log_p0, log_pk))


fit_zip = minimize(nll_zip, x0=[0.0, 0.0], args=(zip_counts,))
pi_hat, lam_hat_zip = expit(fit_zip.x[0]), np.exp(fit_zip.x[1])
lam_pois = zip_counts.mean()
ll_pois = np.sum(stats.poisson.logpmf(zip_counts, lam_pois))
print(f"Poisson MLE : λ̂ = x̄ = {lam_pois:.3f};  log-likelihood = {ll_pois:.1f}")
print(f"ZIP MLE     : π̂ = {pi_hat:.3f}, λ̂ = {lam_hat_zip:.3f};  log-likelihood = {-fit_zip.fun:.1f}   (truth π = 0.4, λ = 3)")
print(f"zeros: observed {np.mean(zip_counts == 0):.3f} | Poisson predicts {np.exp(-lam_pois):.3f} | "
      f"ZIP predicts {pi_hat + (1 - pi_hat) * np.exp(-lam_hat_zip):.3f}")
kk = np.arange(0, 11)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.bar(kk, [np.mean(zip_counts == k) for k in kk], color="0.75", label="observed (400 cells)")
ax.plot(kk, stats.poisson.pmf(kk, lam_pois), "o--", color="tab:red", label=rf"Poisson fit ($\hat\lambda$ = {lam_pois:.2f})")
zip_pmf = (1 - pi_hat) * stats.poisson.pmf(kk, lam_hat_zip)
zip_pmf[0] += pi_hat
ax.plot(kk, zip_pmf, "s-", color="tab:blue", label=rf"ZIP fit ($\hat\pi$ = {pi_hat:.2f}, $\hat\lambda$ = {lam_hat_zip:.2f})")
ax.set_xlabel("molecules detected per cell"); ax.set_ylabel("fraction of cells"); ax.legend()
ax.set_title("SIMULATED dropout: the zero-inflated model explains the excess zeros")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The plain Poisson MLE is forced to explain both the many zeros and the counts around 3 with a
single rate, and compromises at $\hat\lambda =$ 1.83: it predicts only 16 % zeros where 43 % are observed, and
too many 1s and 2s. The ZIP model, fitted numerically, recovers $\hat\pi =$ 0.40 and $\hat\lambda =$ 3.06 (truth 0.4
and 3), predicts the zeros correctly, and its maximised log-likelihood is higher by 148 units. (Real single-cell data are
often fitted well by a negative binomial without zero inflation; whether zero inflation is needed is itself a
model-selection question — [BiologyToML 23](../../../../ml/tutorial/BiologyToML/notebooks/23_model_selection.ipynb).)

✏️ **Your turn.** Refit the ZIP model to only the first 40 cells. How far are $\hat\pi$ and $\hat\lambda$ from the truth now?
Start the optimiser from a few different `x0` values: does it always reach the same maximum?

---
## 9 · Properties of the MLE

> 📍 **Where we are.** Why is maximum likelihood the default? Because, under mild conditions, it has three properties that
> make it a reliable all-purpose estimator — and they connect straight back to the LLN and CLT of Module 22.

1. **Consistency.** As $n\to\infty$, $\hat\theta_n$ converges to the true $\theta$ (in the LLN sense:
   $P(|\hat\theta_n - \theta| < \varepsilon)\to 1$).
2. **Invariance.** If $\hat\theta$ is the MLE of $\theta$, then $g(\hat\theta)$ is the MLE of $g(\theta)$ for any function $g$
   — the MLE of $\sigma$ is $\sqrt{\hat\sigma^2}$, the MLE of the heterozygote frequency $2p(1-p)$ is $2\hat p(1-\hat p)$.
3. **Asymptotic normality.** For large $n$, $\hat\theta \approx \mathcal N\big(\theta, \operatorname{SE}^2\big)$ with
   $\operatorname{SE}\approx 1/\sqrt{-\ell''(\hat\theta)}$: the sharper the peak of the log-likelihood (the more negative its second
   derivative), the smaller the standard error.

For the Bernoulli model, $\ell''(\alpha) = -\frac{s}{\alpha^2} - \frac{n-s}{(1-\alpha)^2}$. At $\hat\alpha = s/n$ (so $s = n\hat\alpha$):

$$-\ell''(\hat\alpha) = \frac{n\hat\alpha}{\hat\alpha^2} + \frac{n(1-\hat\alpha)}{(1-\hat\alpha)^2} = \frac{n}{\hat\alpha} + \frac{n}{1-\hat\alpha}
= \frac{n}{\hat\alpha(1-\hat\alpha)} \;\Longrightarrow\; \operatorname{SE}(\hat\alpha) \approx \sqrt{\frac{\hat\alpha(1-\hat\alpha)}{n}} .$$

*In words:* the curvature formula reproduces the standard error of a proportion from Module 22 — the CLT and the curvature of
the likelihood tell the same story. An approximate 95 % confidence interval is $\hat\theta \pm 1.96\,\operatorname{SE}$
(a **Wald interval**).

In [ ]:
# ── 9.1 Consistency and asymptotic normality of the Bernoulli MLE (true α = 0.3, 20,000 experiments per n) ──
rng_cons = np.random.default_rng(SEED + 91)
alpha_true = 0.3
rows = []
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for n_, col in zip([10, 100, 1000, 10_000], ["tab:red", "tab:orange", "tab:green", "tab:blue"]):
    heads = rng_cons.binomial(n_, alpha_true, 20_000)
    a_hat = heads / n_
    within = np.abs(100 * heads - 30 * n_) < 2 * n_                # |α̂ − 0.3| < 0.02 in exact integer arithmetic
    rows.append({"n": n_, "mean α̂": a_hat.mean(), "SD of α̂": a_hat.std(), "√(α(1−α)/n)": np.sqrt(0.21 / n_),
                 "P(|α̂ − 0.3| < 0.02)": np.mean(within)})
    if n_ >= 100:                                                  # n = 10 gives only 11 possible values (spikes)
        width = {100: 0.01, 1000: 0.003, 10_000: 0.001}[n_]       # whole numbers of possible values per bin
        edges = np.arange(0.1 + 0.5 / n_, 0.5, width)
        ax.hist(a_hat, bins=edges, density=True, histtype="step", color=col, lw=1.5, label=f"n = {n_}")
ax.axvline(alpha_true, color="k", ls="--", lw=1)
ax.set_xlabel(r"MLE $\hat\alpha$"); ax.set_ylabel("density"); ax.legend()
ax.set_title("SIMULATED: the sampling distribution of the MLE concentrates on the truth")
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

🔍 **What just happened?** As $n$ grows from 10 to 10,000 the histogram of $\hat\alpha$ shrinks onto the true 0.3
(consistency: the probability of landing within 0.02 of the truth is about 0.26 for $n = 10$ and $n = 100$, 0.82 for
$n = 1000$ and 1.00 for $n = 10\,000$) and takes the shape of a narrow normal curve whose SD matches $\sqrt{\alpha(1-\alpha)/n}$
(asymptotic normality: 0.0458 simulated vs 0.0458 predicted at $n = 100$, 0.0046 vs 0.0046 at $n = 10\,000$). $n = 10$ is left
out of the plot: its MLE can take only the eleven values $0, 0.1, \dots, 1$ — the asymptotics need a reasonable $n$.

In [ ]:
# ── 9.2 Standard error from curvature, a Wald interval for Example 1, and invariance ──
def bernoulli_curvature_se(s, n):
    """SE of the Bernoulli MLE from the curvature of the log-likelihood at α̂ = s/n."""
    a_hat = s / n
    second_derivative = -s / a_hat ** 2 - (n - s) / (1 - a_hat) ** 2
    return a_hat, 1 / np.sqrt(-second_derivative)


for s_, n_ in [(4, 10), (400, 1000)]:
    a_hat, se_ = bernoulli_curvature_se(s_, n_)
    print(f"s = {s_:3d}, n = {n_:4d}: α̂ = {a_hat:.3f}, SE = 1/√(−ℓ''(α̂)) = {se_:.4f} "
          f"(√(α̂(1−α̂)/n) = {np.sqrt(a_hat * (1 - a_hat) / n_):.4f}); 95 % Wald CI = [{a_hat - 1.96 * se_:.3f}, {a_hat + 1.96 * se_:.3f}]")
# invariance: maximise over σ² directly and compare with (σ̂)² from the σ parametrisation
fit_var = minimize(lambda q, x: -np.sum(stats.norm.logpdf(x, q[0], np.sqrt(np.exp(q[1])))), x0=[7.0, 0.0], args=(x6,))
print(f"invariance: MLE of σ² fitted directly = {np.exp(fit_var.x[1]):.5f};  (MLE of σ)² = {np.exp(2 * fit_n.x[1]):.5f}")
print(f"invariance: MLE of the heterozygote frequency 2p(1 − p) = 2 p̂ (1 − p̂) = {2 * p_hat * (1 - p_hat):.4f}")

🔍 **What just happened?** For Example 1 ($4/10$) the curvature gives $\operatorname{SE} = 0.155$ and a 95 % Wald interval
[0.096, 0.704] — it contains 0.5, so (as Section 4.2 found) ten flips are fully compatible with a fair coin. With 400 heads in
1,000 flips the same MLE 0.4 has SE 0.0155 and the interval [0.370, 0.430] excludes 0.5. Same estimate, very different
certainty: always report a standard error with an MLE. Fitting $\sigma^2$ directly or squaring the fitted $\sigma$ gives the
same number (invariance), and the MLE of the heterozygote frequency is $2\hat p(1-\hat p) =$ 0.429.

### 9.3 Least squares is maximum likelihood

Regression (Module 27) models a response $y_i$ — say, the expression of a target gene — as a straight-line function of a
predictor $x_i$ — the expression of its regulator — plus normal noise: $y_i = \beta_0 + \beta_1 x_i + \varepsilon_i$,
$\varepsilon_i\sim\mathcal N(0, \sigma^2)$ i.i.d. Then $y_i\sim\mathcal N(\beta_0 + \beta_1x_i, \sigma^2)$, and by Section 6 with
$\mu$ replaced by $\beta_0 + \beta_1 x_i$,

$$\ell(\beta_0, \beta_1, \sigma) = -n\log\sqrt{2\pi} - n\log\sigma - \frac{1}{2\sigma^2}\sum_{i=1}^n\big(y_i - \beta_0 - \beta_1 x_i\big)^2 .$$

*In words:* for any fixed $\sigma$, the only part that depends on the $\beta$'s is minus the **sum of squared residuals**, so
maximising the likelihood over $(\beta_0, \beta_1)$ is exactly minimising $\sum_i(y_i - \beta_0 - \beta_1x_i)^2$ — the method of
least squares. The MLE of $\sigma^2$ is again the average squared residual (dividing by $n$).

In [ ]:
# ── 9.3 Regulator → target: least squares (np.polyfit) vs numerical MLE (SIMULATED, true β0 = 1.5, β1 = 0.8, σ = 0.4) ──
rng_reg = np.random.default_rng(SEED + 93)
regulator = rng_reg.normal(6.0, 1.0, 40)
target = 1.5 + 0.8 * regulator + rng_reg.normal(0, 0.4, 40)
b1_ls, b0_ls = np.polyfit(regulator, target, 1)
fit_reg = minimize(lambda q: -np.sum(stats.norm.logpdf(target, q[0] + q[1] * regulator, np.exp(q[2]))), x0=[0.0, 0.0, 0.0])
rss = np.sum((target - b0_ls - b1_ls * regulator) ** 2)
print(f"least squares : β0 = {b0_ls:.4f}, β1 = {b1_ls:.4f}, RSS/n = {rss / 40:.4f}")
print(f"maximum lik.  : β0 = {fit_reg.x[0]:.4f}, β1 = {fit_reg.x[1]:.4f}, σ̂² = {np.exp(2 * fit_reg.x[2]):.4f}")

🔍 **What just happened?** Least squares (`np.polyfit`) and the numerical maximisation of the normal likelihood give the same
intercept and slope to four decimals ($\hat\beta_0 = 1.555$, $\hat\beta_1 = 0.789$; truth 1.5 and 0.8), and the MLE of $\sigma^2$ equals the residual sum of squares divided by $n$.
Least squares is not an arbitrary choice: it is maximum likelihood under Gaussian noise.

### 🌉 Bridge — maximum likelihood everywhere in computational biology

* **Regression and GWAS.** Linear regression is Gaussian MLE (Module 27); a GWAS fits one regression per SNP
  ([BiologyToML 06 · linear regression and GWAS](../../../../ml/tutorial/BiologyToML/notebooks/06_linear_regression_gwas.ipynb)); logistic regression maximises a
  Bernoulli likelihood numerically ([BiologyToML 09](../../../../ml/tutorial/BiologyToML/notebooks/09_logistic_regression.ipynb)).
* **Phylogenetics.** Felsenstein (1981) showed how to compute the likelihood of DNA sequences on a tree; maximum-likelihood
  tree builders search over trees and branch lengths to maximise it — the most widely used way to build phylogenies.
* **Hidden variables.** When some variables are unobserved (cluster labels, hidden states), the likelihood is maximised by
  the EM algorithm ([BiologyToML 13](../../../../ml/tutorial/BiologyToML/notebooks/13_mixture_models_em.ipynb), [20 · HMM learning](../../../../ml/tutorial/BiologyToML/notebooks/20_hmm_learning.ipynb)).
* **The full picture.** [BiologyToML 05 · likelihood, MLE and MAP](../../../../ml/tutorial/BiologyToML/notebooks/05_likelihood_mle_map.ipynb) revisits these ideas from
  the machine-learning side and adds the prior — our next module.

✏️ **Your turn.** For Example 1, compute $-\ell''(\hat p)$ by hand and the 95 % Wald interval. Then compute the *likelihood
interval* $\{p : \ell(p) \ge \ell(\hat p) - 1.92\}$ on a grid and compare. (The cut-off 1.92 is half of $1.96^2$.)

### 📘 Lecture notes — properties of the MLE and summary

**Consistency:** $\hat\theta_n\to\theta$ as $n\to\infty$ (simulated: $P(|\hat\alpha - 0.3| < 0.02)$ = 0.26 at $n = 10$, 1.00
at $n = 10\,000$). **Invariance:** the MLE of $g(\theta)$ is $g(\hat\theta)$ ($\hat\sigma = \sqrt{\hat\sigma^2}$;
$2\hat p(1-\hat p)$). **Asymptotic normality:** $\hat\theta\approx\mathcal N(\theta, \operatorname{SE}^2)$,
$\operatorname{SE}\approx 1/\sqrt{-\ell''(\hat\theta)}$; for Bernoulli $\sqrt{\hat\alpha(1-\hat\alpha)/n}$; Wald CI
$\hat\theta\pm1.96\,\operatorname{SE}$ (Example 1: [0.096, 0.704], includes 0.5). **Not guaranteed:** unbiasedness
($\hat\sigma^2$, $\max_i x_i$ are biased).

**Least squares = Gaussian MLE:** maximising $-\frac{1}{2\sigma^2}\sum_i(y_i - \beta_0 - \beta_1x_i)^2$ over $\beta$ = minimising the
residual sum of squares.

**Numerical MLE:** minimise the negative log-likelihood with `scipy.optimize.minimize` on an unconstrained scale (log, logit);
check against closed forms; needed for zero-inflated, over-dispersed and mixture models.

**Summary.** Statistics makes inferences based on data, in three phases (collecting, describing, analysing). Maximum likelihood
estimation is the most commonly used technique for learning a parametric probability model from data.

> 🎓 **Exam tip.** An MLE without an uncertainty is half an answer. Be able to get the SE from the second derivative and
> to explain in one sentence why more data give a sharper log-likelihood peak.

## 📌 Key takeaways

- **Statistical inference** chooses a member $\hat f$ of a parametric model $\mathcal F = \{f(x\mid\theta):\theta\in\Theta\}$ that
  explains data from an unseen data generator: observations + knowledge and assumptions + optimisation.
- **Choose the family with knowledge and exploration**: counts → Poisson (check variance ≈ mean), bell-shaped → normal,
  over-dispersed counts → negative binomial.
- The **likelihood** of i.i.d. data is $p(\mathcal D\mid\theta) = \prod_i p(x_i\mid\theta)$, a function of $\theta$ but not a
  probability distribution over $\theta$; we maximise its **log**, which turns products into sums and avoids underflow.
- **MLE** = $\arg\max_\theta p(\mathcal D\mid\theta)$: differentiate, solve, check it is a maximum, and look at the boundary
  when there is no critical point (all heads → 1; Uniform$(0,\theta)$ → $\max_i x_i$).
- Closed forms: Bernoulli $s/n$ (Example 1: 0.4; allele counting $(2n_2 + n_1)/2n$), normal $\bar x$ and
  $\frac1n\sum(x_i - \bar x)^2$ (biased by $(n-1)/n$), Poisson $\bar x$ (with exposures $\sum k/\sum T$), exponential $1/\bar x$.
- Without a closed form, **minimise the negative log-likelihood numerically** (zero-inflated Poisson for dropout).
- The MLE is **consistent**, **invariant** and **asymptotically normal** with SE $\approx 1/\sqrt{-\ell''(\hat\theta)}$ — and an
  estimate of 0.4 from 10 flips is no evidence of an unfair coin. **Least squares is Gaussian MLE.**

---
## ✅ Checkpoint

1. Isolates R, S, S, S, R, S (R = 1). Write $L(\alpha)$ and $\ell(\alpha)$ and find $\hat\alpha$ with the maximum check.
   <details><summary>Answer</summary>$L = \alpha^2(1-\alpha)^4$; $\ell = 2\log\alpha + 4\log(1-\alpha)$; $\ell' = 2/\alpha - 4/(1-\alpha) = 0 \Rightarrow 2(1-\alpha) = 4\alpha \Rightarrow \hat\alpha = 1/3$; $\ell'' = -2/\alpha^2 - 4/(1-\alpha)^2 < 0$.</details>
2. Why does the binomial coefficient $\binom{10}{4}$ not change the MLE in Example 1? Which probability does it belong to?
   <details><summary>Answer</summary>It is a positive constant (no $p$), so it scales $L$ without moving its maximum (in logs it is an additive constant with derivative 0). $\binom{10}{4}p^4(1-p)^6$ is the probability of "4 heads in 10 flips in any order"; the specific sequence has probability $p^4(1-p)^6$.</details>
3. Give two reasons to maximise $\log L$ instead of $L$.
   <details><summary>Answer</summary>(1) $\log$ is increasing, so the maximiser is the same, and the log turns the product into a sum that is easy to differentiate. (2) Products of many probabilities underflow to 0 in floating point (2,000 densities, $0.5^{2000}$); sums of logs do not.</details>
4. Ten of ten patients respond to a drug. What is the MLE of the response probability, and why is there no critical point?
   <details><summary>Answer</summary>$L(\alpha) = \alpha^{10}$, $L' = 10\alpha^9 > 0$ on $(0,1)$: no zero inside, the likelihood increases to the boundary, so $\hat\alpha = 1$ (the maximum allowable value in the direction of increase).</details>
5. Derive the Poisson MLE for counts $x_1,\dots,x_n$ and compute it for 3, 0, 2, 5, 0.
   <details><summary>Answer</summary>$\ell = -n\lambda + (\sum x_i)\log\lambda - \sum\log x_i!$; $\ell' = -n + \sum x_i/\lambda = 0 \Rightarrow \hat\lambda = \bar x = 10/5 = 2$.</details>
6. For expression values 2, 4, 6 compute $\hat\mu$, $\hat\sigma^2_{\text{MLE}}$ and $s^2$. Which is unbiased?
   <details><summary>Answer</summary>$\hat\mu = 4$; $\hat\sigma^2 = (4 + 0 + 4)/3 = 8/3 \approx 2.67$; $s^2 = 8/2 = 4$. $s^2$ is unbiased; $E[\hat\sigma^2] = \frac{n-1}{n}\sigma^2$.</details>
7. In the normal derivation, what is $\frac{d}{d\sigma}\big[-\frac{1}{2\sigma^2}\sum(x_i-\mu)^2\big]$? Solve $-\frac n\sigma + \sigma^{-3}\sum(x_i-\mu)^2 = 0$ for $\sigma^2$.
   <details><summary>Answer</summary>$+\sigma^{-3}\sum(x_i-\mu)^2$ (since $\frac{d}{d\sigma}\sigma^{-2} = -2\sigma^{-3}$). Multiplying by $\sigma^3$: $\sum(x_i-\mu)^2 = n\sigma^2$, so $\hat\sigma^2 = \frac1n\sum(x_i-\hat\mu)^2$.</details>
8. Genotype counts $n_0 = 60$, $n_1 = 35$, $n_2 = 5$. Estimate the allele frequency and the expected number of heterozygotes under HWE.
   <details><summary>Answer</summary>$\hat p = (2\cdot5 + 35)/200 = 0.225$; expected heterozygotes $100\cdot2\cdot0.225\cdot0.775 = 34.9$ (by invariance, the MLE of $2p(1-p)$ is $2\hat p(1-\hat p) = 0.349$).</details>
9. Four lines accumulate 3, 5, 2, 6 mutations with exposures $1, 2, 1, 4\times10^{10}$ base-generations. Give $\hat\mu$ and explain why it is not the average of $k_i/T_i$.
   <details><summary>Answer</summary>$\hat\mu = 16/(8\times10^{10}) = 2\times10^{-10}$. The MLE weights lines by their exposure (total events / total exposure); averaging per-line rates gives equal weight to short, noisy lines.</details>
10. Residence times 1.0, 3.0, 2.0, 6.0 s. Give the exponential MLE of the off-rate and of the mean residence time.
    <details><summary>Answer</summary>$\bar x = 3$ s, $\hat k = 1/3$ s⁻¹; by invariance the MLE of the mean $1/k$ is $3$ s.</details>
11. Use the curvature to get the SE of $\hat\alpha$ for 4 heads in 10 flips, and the 95 % Wald interval. Does it contain 0.5?
    <details><summary>Answer</summary>$-\ell''(\hat\alpha) = n/(\hat\alpha(1-\hat\alpha)) = 10/0.24 = 41.67$; SE $= 0.155$; $0.4 \pm 0.304 = [0.096, 0.704]$ — contains 0.5.</details>
12. Explain in two sentences why least-squares regression is maximum likelihood.
    <details><summary>Answer</summary>If $y_i = \beta_0 + \beta_1x_i + \varepsilon_i$ with i.i.d. $\mathcal N(0,\sigma^2)$ noise, the log-likelihood is a constant minus $\frac{1}{2\sigma^2}\sum(y_i - \beta_0 - \beta_1x_i)^2$. Maximising it over the $\beta$'s is therefore the same as minimising the sum of squared residuals.</details>

## 🧠 Questions & discussion

1. The coin-flip simulator emits a nonsense number once in 1,000 outputs. If we fit a Bernoulli model and silently drop the
   glitches, what are we assuming about them? In sequencing, what would be the danger of dropping all "N" calls?
2. Section 2 chose Poisson because variance ≈ mean. Real single-cell data are over-dispersed. Discuss biological sources of
   extra variance (bursting, cell cycle, cell size) and technical ones (capture efficiency, amplification).
3. The MLE is "the parameter value under which the data were most probable". Why is that not the same as "the most probable
   parameter value given the data"? What extra ingredient would you need (Module 24)?
4. An MLE can sit on the boundary (all patients responded → $\hat\alpha = 1$). Would you report a 100 % response rate? What
   would a clinician want instead?
5. Maximum likelihood requires a model. When might a nonparametric estimate (Module 22's bootstrap, an empirical
   distribution) be preferable, and what do you lose?

## 🚀 Bonus — going further

- **Fisher information.** $I(\theta) = -E[\ell''(\theta)]$ for one observation; the Cramér–Rao bound says no unbiased estimator
  has variance below $1/(nI(\theta))$. Compute $I(\lambda)$ for the Poisson and show that $\bar x$ attains the bound.
- **Likelihood-ratio intervals and tests.** $2[\ell(\hat\theta) - \ell(\theta_0)]$ is approximately $\chi^2_1$ under
  $H_0: \theta = \theta_0$ (Wilks' theorem) — the bridge from MLE to hypothesis testing (Modules 25–26).
- **Negative binomial MLE.** Fit a negative binomial (mean and dispersion) to the over-dispersed T-cell counts of Section 2 with
  `minimize` and compare its maximised log-likelihood with the Poisson fit.
- **Method of moments.** Estimate the ZIP parameters by matching the sample mean and the fraction of zeros, and compare the
  sampling variability of these estimates with the MLE over 1,000 simulated data sets.
- **EM for allele frequencies.** Derive the EM algorithm for allele frequencies from genotype likelihoods (Section 5's
  Bridge).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            23 · Parameter estimation and maximum likelihood
Input:             SIMULATED single-cell counts (simulate_cell_counts), genotypes (simulate_genotypes), glitchy coin /
                   base-call outputs, expression values, mutation-accumulation lines, residence times, foci positions,
                   dropout counts, regulator–target pairs; typed-in Example 1 (1,1,1,0,0,0,0,0,1,0)
Output:            likelihood curves and surfaces, closed-form and numerical MLEs, bias table, SEs and Wald intervals
Method:            likelihood = Π f(x_i | θ) → log → derivative = 0 → maximum check / boundary; minimize(−ℓ) otherwise
Key equation:      θ̂ = argmax_θ Σ_i log f(x_i | θ);  α̂ = s/n;  μ̂ = x̄, σ̂² = (1/n)Σ(x_i − x̄)²;  λ̂ = x̄;  SE ≈ 1/√(−ℓ''(θ̂))
Key parameters:    model family; parametrisation (σ vs σ², logit, log); sample size n
What changed when I modified …  (e.g. n = 10 → 1000 flips; dropout π = 0.4 → 0; ZIP fitted on 40 cells)
One failure mode:  wrong model family (Poisson for over-dispersed or zero-inflated counts) → biased, over-confident MLEs
Connection to 02-680: statistical inference framework, parametric models, data likelihood, log-likelihood, MLE recipe,
                   Example 1, Bernoulli / normal / Poisson MLE derivations
```

## 📚 References

- Wasserman L. (2004) *All of Statistics: A Concise Course in Statistical Inference*. Springer — Ch. 6 (models, statistical
  inference and learning), Ch. 9 (parametric inference: maximum likelihood, consistency, invariance, asymptotic normality,
  Fisher information).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — §7.5 (maximum likelihood estimators),
  §7.6 (properties of MLEs).
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press — §8.3
  (parameter estimation: maximum likelihood), §9.2 (linear regression as maximum likelihood).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences*. CRC Press — the chapters on estimation.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Felsenstein J. (1981) Evolutionary trees from DNA sequences: a maximum likelihood approach. *Journal of Molecular Evolution*
  17:368–376.

## ➡️ Next

**[Module 24 · 🧭 Bayesian estimation and maximum a posteriori](24_bayesian_estimation_map.ipynb)** — maximum likelihood ignores everything we knew before the experiment, and it shows: ten responders out
of ten give $\hat\alpha = 1$, and a rare allele never seen in 50 people gets frequency 0. Module 24 brings back the prior that
Bayes' theorem put next to the likelihood in Section 3: posterior ∝ likelihood × prior. It introduces pseudocounts, the Beta
and Gamma priors, the **maximum a posteriori** (MAP) estimate and the posterior distribution — and shows that the MLEs of this
module are what Bayesian estimates become as the data grow.